# Обучение нейросети синтеза ДОЭ (развёрнутая сеть v2) — Kaggle / Colab

Этот ноутбук воспроизводит обучение весов `weights/doe_unrolled_v2.pt` из репозитория.
Исходники проекта вшиты в ноутбук, ничего дополнительно загружать не нужно.

**Данные для обучения:** [Tiny-ImageNet на HuggingFace — `zh-plus/tiny-imagenet`](https://huggingface.co/datasets/zh-plus/tiny-imagenet)
(110 000 фото 64×64: части `train` + `valid`). Скачиваются автоматически. Готовые ДОЭ не нужны:
сеть учится через физическую модель дифракции Френеля.

**Kaggle, обучение в фоне:**
1. Справа *Settings*: **Accelerator → GPU T4 x2**, **Internet → On**.
2. **Save Version → Save & Run All (Commit) → Save**. Вкладку можно закрыть.
3. Через ~11 ч: вкладка **Output** → `doe_results.zip` (веса, лог обучения, результаты на примерах, набор данных).

Обучение само подбирает число итераций под `HOURS` и завершается вовремя (лимит сессии Kaggle — 12 ч).
Исходные веса получены именно так: T4 x2, `HOURS = 10.5`, 11 903 итерации, 9,6 ч.
Дообучение поверх готового: *Add Input → Notebook Output →* прошлая версия ноутбука — прошлый `best.pt`
найдётся автоматически (тёплый старт).

In [ ]:
import os, sys, glob
ROOT = '/kaggle/working/doe_nn' if os.path.exists('/kaggle/working') else '/content/doe_nn'
DATA_DIR = '/tmp/doe_data'          # кэш датасета не попадает в Output (он большой)
HOURS = 10.5                        # бюджет на обучение, часов (Kaggle: всего 12 ч на сессию)
for d in ('doe', 'scripts'):
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)
os.chdir(ROOT)
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'
import subprocess

def run(cmd, strict=True):
    # Запуск команды с выводом в реальном времени. strict=True: при ошибке ноутбук
    # останавливается (не тратим GPU-квоту зря); strict=False: только предупреждение.
    print('$', cmd, flush=True)
    pr = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                          encoding='utf-8', errors='replace')
    for line in pr.stdout:
        print(line, end='', flush=True)
    code = pr.wait()
    if code != 0:
        msg = f'команда завершилась с ошибкой (код {code}): {cmd}'
        if strict:
            raise RuntimeError(msg)
        print('ВНИМАНИЕ:', msg)
    return code == 0

print('Рабочая папка:', ROOT)
!nvidia-smi -L || echo "GPU не найден - включите GPU в настройках среды"

## 1. Исходники проекта (записываются в файлы)

In [ ]:
%%writefile doe/__init__.py
"""Нейросетевой синтез фазовых ДОЭ: физика, данные, модель, метрики."""

In [ ]:
%%writefile doe/physics.py
"""Физическая модель: дифракция Френеля «с одним преобразованием Фурье».

K(x,y) = exp(jkz)/(j*lambda*z) * exp(j*pi/(lambda*z)*(x^2+y^2))
         * FFT{ H(u,v) * exp(j*pi/(lambda*z)*(u^2+v^2)) }

H(u,v) = exp(j*phi(u,v)) - поле сразу за фазовым ДОЭ (освещение плоской волной).
Множитель перед FFT и выходная квадратичная фаза не меняют модуль |K|,
поэтому при обучении считаем только |FFT{H * Q_in}|; в numpy-версии
(fresnel_numpy) формула реализована полностью, как на слайде.
"""
import math

import numpy as np
import torch

# ---- Параметры задачи -------------------------------------------------------
N = 128                      # размер ДОЭ и изображений, пикселов
DX = 10e-6                   # шаг пиксела ПВМС, м
WAVELENGTH = 532e-9          # длина волны, м
DISTANCES = (24.1e-3, 25.3e-3, 26.5e-3, 27.7e-3)   # плоскости восстановления, м

# Информативные области (строка, столбец). В задании написано [0:61], [66:127];
# в выданных картинках ненулевые пиксели занимают индексы 0..61 и 66..127
# включительно (62 px), поэтому по умолчанию берём включительные границы.
ROI_INCLUSIVE = [
    (slice(0, 62), slice(0, 62)),
    (slice(0, 62), slice(66, 128)),
    (slice(66, 128), slice(0, 62)),
    (slice(66, 128), slice(66, 128)),
]
# Буквальное прочтение как Python-срезы (61 px). Считаем метрики и так, для сверки.
ROI_LITERAL = [
    (slice(0, 61), slice(0, 61)),
    (slice(0, 61), slice(66, 127)),
    (slice(66, 127), slice(0, 61)),
    (slice(66, 127), slice(66, 127)),
]
ROI = ROI_INCLUSIVE
ROI_SIZE = 62


def output_pixel_pitch(z, n=N, dx=DX, wl=WAVELENGTH):
    """Шаг пиксела в плоскости восстановления для метода с одним БПФ."""
    return wl * z / (n * dx)


def chirp_nyquist_ratio(z, n=N, dx=DX, wl=WAVELENGTH):
    """>= 1 означает, что квадратичная фаза exp(j*pi*r^2/(lambda*z)) дискретизована
    без наложения спектров (макс. набег фазы между соседними пикселами <= pi)."""
    return wl * z / (n * dx * dx)


# ---- Кодирование фазы в 8-битное изображение ---------------------------------
def wrap_phase(phi):
    """Приводит фазу к диапазону [-pi, pi)."""
    if isinstance(phi, torch.Tensor):
        return torch.remainder(phi + math.pi, 2 * math.pi) - math.pi
    return np.mod(phi + np.pi, 2 * np.pi) - np.pi


def phase_to_uint8(phi):
    """-pi -> 0, +pi -> 255 (как требует задание)."""
    phi = wrap_phase(np.asarray(phi, dtype=np.float64))
    return np.clip(np.round((phi + np.pi) / (2 * np.pi) * 255.0), 0, 255).astype(np.uint8)


def uint8_to_phase(img):
    return np.asarray(img, dtype=np.float64) / 255.0 * 2 * np.pi - np.pi


# ---- Эталонная numpy-реализация (полная формула со слайда) --------------------
def fresnel_numpy(H, z, dx=DX, wl=WAVELENGTH, fftshift=True):
    """Полная формула Френеля с одним БПФ. H - комплексное поле (n, n)."""
    n = H.shape[-1]
    k = 2 * np.pi / wl
    c = (np.arange(n) - n / 2) * dx
    u, v = np.meshgrid(c, c, indexing="ij")
    dxo = wl * z / (n * dx)                       # шаг в плоскости восстановления
    co = (np.arange(n) - n / 2) * dxo
    x, y = np.meshgrid(co, co, indexing="ij")
    q_in = np.exp(1j * np.pi / (wl * z) * (u ** 2 + v ** 2))
    spec = np.fft.fft2(np.fft.ifftshift(H * q_in))
    if fftshift:
        spec = np.fft.fftshift(spec)
    pref = np.exp(1j * k * z) / (1j * wl * z)
    q_out = np.exp(1j * np.pi / (wl * z) * (x ** 2 + y ** 2))
    return pref * q_out * spec * dx * dx          # dx^2 - мера интегрирования


def reconstruct_numpy(phi, distances=DISTANCES, fftshift=True):
    """Модули восстановленных полей для всех плоскостей: (P, n, n)."""
    H = np.exp(1j * np.asarray(phi, dtype=np.float64))
    return np.stack([np.abs(fresnel_numpy(H, z, fftshift=fftshift)) for z in distances])


# ---- Дифференцируемая torch-реализация (для обучения) ------------------------
class Fresnel(torch.nn.Module):
    """Прямое и обратное распространение ДОЭ -> P плоскостей (батчево, на GPU)."""

    def __init__(self, n=N, dx=DX, wl=WAVELENGTH, distances=DISTANCES, fftshift=True):
        super().__init__()
        self.fftshift = fftshift
        self.n_planes = len(distances)
        c = (torch.arange(n, dtype=torch.float64) - n / 2) * dx
        u, v = torch.meshgrid(c, c, indexing="ij")
        r2 = u ** 2 + v ** 2
        q = torch.stack([torch.exp(1j * math.pi / (wl * z) * r2) for z in distances]).to(torch.complex64)
        # Храним как вещественный (P, n, n, 2): так буфер надёжно копируется между
        # несколькими GPU; в чекпойнт не пишется (это константа, считается заново).
        self.register_buffer("q_ri", torch.view_as_real(q).contiguous(), persistent=False)

    @property
    def q(self):
        return torch.view_as_complex(self.q_ri)                   # (P, n, n) complex64

    def _fft(self, x):
        x = torch.fft.fft2(torch.fft.ifftshift(x, dim=(-2, -1)), norm="ortho")
        return torch.fft.fftshift(x, dim=(-2, -1)) if self.fftshift else x

    def _ifft(self, x):
        if self.fftshift:
            x = torch.fft.ifftshift(x, dim=(-2, -1))
        return torch.fft.fftshift(torch.fft.ifft2(x, norm="ortho"), dim=(-2, -1))

    def forward_field(self, h):
        """h: комплексное поле ДОЭ (B, n, n) -> поля в плоскостях (B, P, n, n)."""
        return self._fft(h.unsqueeze(1) * self.q)

    def forward_phase(self, phi):
        """phi: (B, n, n) -> модули полей |K| (B, P, n, n)."""
        return self.forward_field(torch.polar(torch.ones_like(phi), phi)).abs()

    def backward_field(self, k):
        """Обратное распространение: поля в плоскостях (B, P, n, n) -> поля в
        плоскости ДОЭ, по отдельности для каждой плоскости (B, P, n, n)."""
        return self._ifft(k) * self.q.conj()


def roi_crop(x, rois=ROI):
    """x: (..., P, n, n) -> (..., P, h, w): из плоскости k берём область k."""
    return torch.stack([x[..., k, r, c] for k, (r, c) in enumerate(rois)], dim=-3)


def roi_crop_np(x, rois=ROI):
    return np.stack([x[k][r, c] for k, (r, c) in enumerate(rois)])

In [ ]:
%%writefile doe/metrics.py
"""Метрики (КК Пирсона, ИСС/SSIM) и функция потерь."""
import numpy as np
import torch
import torch.nn.functional as F


def _gauss_window(size=11, sigma=1.5, device=None, dtype=torch.float32):
    g = torch.arange(size, dtype=dtype, device=device) - (size - 1) / 2
    g = torch.exp(-(g ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    return (g[:, None] * g[None, :])[None, None]


def ssim_torch(a, b, data_range=1.0, win_size=11, sigma=1.5, k1=0.01, k2=0.03, pad="valid"):
    """SSIM (Wang et al., 2004) с гауссовым окном 11x11, sigma=1.5.
    pad="valid": окна только целиком внутри картинки - как skimage(gaussian_weights=True,
                 use_sample_covariance=False) [основной вариант];
    pad="replicate": края дополняются повтором, среднее по всем пикселам - как ssim() в MATLAB.
    a, b: (..., H, W). Возвращает (...)."""
    shape = a.shape
    a = a.reshape(-1, 1, *shape[-2:])
    b = b.reshape(-1, 1, *shape[-2:])
    if pad == "replicate":
        r = win_size // 2
        a = F.pad(a, (r, r, r, r), mode="replicate")
        b = F.pad(b, (r, r, r, r), mode="replicate")
    w = _gauss_window(win_size, sigma, a.device, a.dtype)
    c1, c2 = (k1 * data_range) ** 2, (k2 * data_range) ** 2
    mu_a, mu_b = F.conv2d(a, w), F.conv2d(b, w)
    saa = F.conv2d(a * a, w) - mu_a ** 2
    sbb = F.conv2d(b * b, w) - mu_b ** 2
    sab = F.conv2d(a * b, w) - mu_a * mu_b
    s = ((2 * mu_a * mu_b + c1) * (2 * sab + c2)) / ((mu_a ** 2 + mu_b ** 2 + c1) * (saa + sbb + c2))
    return s.mean(dim=(-3, -2, -1)).reshape(shape[:-2])


def pcc_torch(a, b, eps=1e-8):
    """Коэффициент корреляции Пирсона по последним двум осям."""
    a = a - a.mean(dim=(-2, -1), keepdim=True)
    b = b - b.mean(dim=(-2, -1), keepdim=True)
    num = (a * b).sum(dim=(-2, -1))
    den = torch.sqrt((a * a).sum(dim=(-2, -1)) * (b * b).sum(dim=(-2, -1)) + eps)
    return num / den


def ls_scale(amp, tgt, eps=1e-8):
    """Масштаб s, минимизирующий ||s*amp - tgt||^2 (подбор «экспозиции»)."""
    return (amp * tgt).sum(dim=(-2, -1), keepdim=True) / ((amp * amp).sum(dim=(-2, -1), keepdim=True) + eps)


def doe_loss(amp, tgt, w_mse=1.0, w_pcc=1.0, w_ssim=1.0):
    """amp, tgt: (B, P, h, w) - модуль восстановленного поля и целевая амплитуда
    в информативных областях (tgt в [0, 1]). Масштаб amp подбирается
    аналитически, поэтому потеря зависит только от формы распределения."""
    a = amp * ls_scale(amp, tgt)
    mse = ((a - tgt) ** 2).mean()
    pcc = pcc_torch(a, tgt).mean()
    ssim = ssim_torch(a, tgt, data_range=1.0).mean()
    loss = w_mse * mse * 10.0 + w_pcc * (1 - pcc) + w_ssim * (1 - ssim)
    return loss, {"mse": mse.item(), "pcc": pcc.item(), "ssim": ssim.item()}


def efficiency(amp_full, amp_roi):
    """Дифракционная эффективность: доля энергии плоскости k, попавшая в её
    информативную область. amp_full (B,P,n,n), amp_roi (B,P,h,w) -> (B,P)."""
    return (amp_roi ** 2).sum(dim=(-2, -1)) / ((amp_full ** 2).sum(dim=(-2, -1)) + 1e-12)


def frame_norm_loss(amp_full, tgt_roi, rois):
    """Та же ошибка, но восстановление нормировано на максимум ВСЕГО кадра каждой
    плоскости (как сделал бы mat2gray / a / a.max()), без подбора яркости.
    Заставляет держать свет вне информативных областей не ярче нужного уровня -
    тогда метрики не зависят от того, как жюри нормирует восстановление."""
    from .physics import roi_crop
    a = roi_crop(amp_full / (amp_full.amax(dim=(-2, -1), keepdim=True) + 1e-8), rois)
    return 10 * ((a - tgt_roi) ** 2).mean() + (1 - pcc_torch(a, tgt_roi).mean()) \
        + (1 - ssim_torch(a, tgt_roi, data_range=1.0).mean())


def full_loss(amp_full, tgt_roi, rois, w_eff=0.0, w_frame=0.0):
    """doe_loss + необязательный штраф за низкую эффективность (w_eff > 0
    заставляет держать свет внутри областей ценой небольшой потери метрик)
    + необязательная ошибка при нормировке по всему кадру (w_frame > 0,
    используется в доводке для устойчивости к способу нормировки у жюри)."""
    from .physics import roi_crop
    amp_roi = roi_crop(amp_full, rois)
    loss, m = doe_loss(amp_roi, tgt_roi)
    if w_frame > 0:
        loss = loss + w_frame * frame_norm_loss(amp_full, tgt_roi, rois)
    if w_eff > 0:
        eff = efficiency(amp_full, amp_roi).mean()
        loss = loss + w_eff * (1 - eff)
        m["eff"] = eff.item()
    return loss, m


# ---- Метрики для итогового отчёта (по 8-битным картинкам) ---------------------
def pcc_np(a, b):
    a = np.array(a, np.float64).ravel()      # копия: не портим входной массив
    b = np.array(b, np.float64).ravel()
    a = a - a.mean()
    b = b - b.mean()
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum() + 1e-12))


def ssim_np(a, b, data_range=255.0, pad="valid"):
    """SSIM на гауссовом окне (та же реализация, что и при обучении)."""
    ta = torch.as_tensor(np.asarray(a, np.float64))
    tb = torch.as_tensor(np.asarray(b, np.float64))
    return float(ssim_torch(ta, tb, data_range=data_range, pad=pad))


def ssim_global_np(a, b, data_range=255.0, k1=0.01, k2=0.03):
    """ИСС ровно по формуле со слайда 3: средние, дисперсии и ковариация
    считаются по всей сравниваемой области сразу (одно «окно» на всю область)."""
    a = np.array(a, np.float64).ravel()
    b = np.array(b, np.float64).ravel()
    c1, c2 = (k1 * data_range) ** 2, (k2 * data_range) ** 2
    ma, mb = a.mean(), b.mean()
    va, vb = a.var(), b.var()
    cov = ((a - ma) * (b - mb)).mean()
    return float((2 * ma * mb + c1) * (2 * cov + c2) / ((ma ** 2 + mb ** 2 + c1) * (va + vb + c2)))


def ssim_skimage(a, b, data_range=255.0):
    """Сверка с библиотекой scikit-image, если она установлена. Возвращает два
    варианта: с гауссовым окном (как MATLAB) и с настройками skimage по умолчанию."""
    try:
        from skimage.metrics import structural_similarity as ss
    except ImportError:
        return None
    a = np.asarray(a, np.float64)
    b = np.asarray(b, np.float64)
    g = ss(a, b, data_range=data_range, gaussian_weights=True, sigma=1.5, use_sample_covariance=False)
    d = ss(a, b, data_range=data_range)
    return {"ssim_skimage_gauss": float(g), "ssim_skimage_default": float(d)}

In [ ]:
%%writefile doe/model.py
"""Нейросеть синтеза ДОЭ с физическим слоем внутри (physics-in-the-loop).

Схема (идея близка к DeepCGH, Eybposh et al., Opt. Express 2020):
  4 целевых изображения (4x128x128)
    -> U-Net A: предсказывает фазу поля psi_k в каждой из 4 плоскостей
    -> поля T_k*exp(j*psi_k) обратно распространяются к ДОЭ (точный оператор
       Френеля, не обучается) и складываются с обучаемыми весами
    -> phi0 = arg(сумма)  - первое приближение (как 1 шаг «умного» GS)
    -> U-Net B: по cos(phi0), sin(phi0), |сумма| и целям предсказывает поправку
    -> phi = phi0 + pi*tanh(delta)  - итоговая фаза ДОЭ (фазовый, |H| = 1)
Сеть обучается без готовых голограмм: потеря считается после прямого
распространения phi в 4 плоскости (самообучение через физическую модель).
"""
import torch
import torch.nn as nn

from .physics import Fresnel


def _gn(c):
    g = next(g for g in (8, 6, 4, 3, 2, 1) if c % g == 0)
    return nn.GroupNorm(g, c)


class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), _gn(cout), nn.SiLU(),
            nn.Conv2d(cout, cout, 3, padding=1), _gn(cout), nn.SiLU())
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        return self.net(x) + self.skip(x)


class UNet(nn.Module):
    def __init__(self, cin, cout, base=32, depth=4):
        super().__init__()
        ch = [base * 2 ** i for i in range(depth + 1)]
        self.inc = ConvBlock(cin, ch[0])
        self.down = nn.ModuleList([ConvBlock(ch[i], ch[i + 1]) for i in range(depth)])
        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in reversed(range(depth))])
        self.dec = nn.ModuleList([ConvBlock(2 * ch[i], ch[i]) for i in reversed(range(depth))])
        self.pool = nn.AvgPool2d(2)
        self.out = nn.Conv2d(ch[0], cout, 1)

    def forward(self, x):
        skips = [self.inc(x)]
        for d in self.down:
            skips.append(d(self.pool(skips[-1])))
        x = skips.pop()
        for up, dec in zip(self.up, self.dec):
            x = dec(torch.cat([up(x), skips.pop()], 1))
        return self.out(x)


def _unit(z, eps=1e-6):
    """Единичный фазор z/|z| с гладким градиентом (без atan2)."""
    return z / torch.sqrt(z.real ** 2 + z.imag ** 2 + eps)


class DOENet(nn.Module):
    def __init__(self, base=32, base_ref=24, depth=4, n_planes=4, fftshift=True):
        super().__init__()
        self.prop = Fresnel(fftshift=fftshift)
        self.n_planes = n_planes
        self.unet_a = UNet(n_planes, 2 * n_planes, base, depth)
        self.unet_b = UNet(3 + n_planes, 1, base_ref, depth)
        self.plane_w = nn.Parameter(torch.ones(n_planes))
        nn.init.zeros_(self.unet_b.out.weight)      # старт: phi = phi0
        nn.init.zeros_(self.unet_b.out.bias)

    def forward(self, tgt):
        """tgt: (B, P, n, n) целевые амплитуды (>=0). Возвращает
        (phasor, phasor0): единичные фазоры итогового ДОЭ и первого приближения,
        оба (B, n, n), complex64. Фаза = torch.angle(phasor)."""
        B, P, n, _ = tgt.shape
        t = tgt / (tgt.amax(dim=(-2, -1), keepdim=True) + 1e-6)
        o = self.unet_a(t).float().view(B, P, 2, n, n)
        e = _unit(torch.complex(o[:, :, 0], o[:, :, 1]))
        h = (self.prop.backward_field(t * e) * self.plane_w.view(1, P, 1, 1)).sum(1)
        h = h / (h.abs().pow(2).mean(dim=(-2, -1), keepdim=True).sqrt() + 1e-8)
        p0 = _unit(h)
        x = torch.cat([p0.real.unsqueeze(1), p0.imag.unsqueeze(1),
                       h.abs().unsqueeze(1).clamp(max=4.0), t], 1)
        d = torch.pi * torch.tanh(self.unet_b(x).float()[:, 0])
        p = p0 * torch.polar(torch.ones_like(d), d)
        return p, p0

    def forward_all(self, tgt):
        """Все промежуточные ответы (для глубокого контроля): [phasor0, phasor]."""
        p, p0 = self.forward(tgt)
        return [p0, p]

    def forward_phases(self, tgt):
        """То же, но фазами (вещественные тензоры) - удобно для нескольких GPU."""
        return [_safe_angle(p) for p in self.forward_all(tgt)]

    @torch.no_grad()
    def predict_phase(self, tgt):
        p, _ = self.forward(tgt)
        return torch.angle(p)

    def amplitudes(self, phasor):
        """Модули полей в 4 плоскостях для фазора ДОЭ (B, n, n)."""
        return self.prop.forward_field(phasor).abs()


# =============================================================================
# v2: развёрнутая (unrolled) сеть - «выученный итеративный алгоритм»
# =============================================================================
def _safe_angle(z, eps=1e-12):
    """arg(z) без бесконечного градиента в нуле."""
    return torch.atan2(z.imag, z.real + eps * (z.abs() < eps))


class UnrolledDOENet(nn.Module):
    """Идея (deep unrolling / «обучаемый Герчберг-Сэкстон»):

      phi_0 = начальное приближение (как в v1: U-Net A -> обратный Френель)
      для t = 1..T (каждый блок со своими весами):
          K_k   = Френель(phi)                    4 плоскости, точная физика
          GS-шаг: в информативной области модуль -> цель, обратно к ДОЭ -> h_gs
          невязка: (|K|-цель) с фазой K, обратно к ДОЭ -> g  (≈ градиент ошибки)
          CNN_t(cos/sin шага GS, |h_gs|, невязка g, cos/sin phi) -> delta
          phi <- phi + alpha_t * (шаг GS) + pi*tanh(delta)

    Без обучения (delta=0, alpha=1) блок - ровно одна итерация GS, т.е. сеть
    заведомо не хуже T итераций классики; обучение учит делать шаги намного
    «умнее». Всё дифференцируемо, сеть учится end-to-end через физику.
    """

    N_FEAT = 7

    def __init__(self, base=32, stage_base=32, stages=12, depth=3, depth_a=4, n_planes=4, fftshift=True,
                 grad_ckpt=True):
        super().__init__()
        from .physics import N, ROI
        self.prop = Fresnel(fftshift=fftshift)
        self.n_planes = n_planes
        self.grad_ckpt = grad_ckpt
        self.unet_a = UNet(n_planes, 2 * n_planes, base, depth_a)
        self.plane_w = nn.Parameter(torch.ones(n_planes))
        self.blocks = nn.ModuleList([UNet(self.N_FEAT, 1, stage_base, depth) for _ in range(stages)])
        for b in self.blocks:                      # старт: каждый блок = шаг GS
            nn.init.zeros_(b.out.weight)
            nn.init.zeros_(b.out.bias)
        self.alpha = nn.Parameter(torch.ones(stages))
        m = torch.zeros(n_planes, N, N, dtype=torch.bool)
        for k, (r, c) in enumerate(ROI):
            m[k, r, c] = True
        self.register_buffer("mask", m)
        self.rois = ROI

    def _init_phase(self, t):
        B, P, n, _ = t.shape
        o = self.unet_a(t).float().view(B, P, 2, n, n)
        e = _unit(torch.complex(o[:, :, 0], o[:, :, 1]))
        h = (self.prop.backward_field(t * e) * self.plane_w.view(1, P, 1, 1)).sum(1)
        return _safe_angle(h)

    def _physics(self, phi, t):
        from .metrics import ls_scale
        from .physics import roi_crop
        P = self.n_planes
        p = torch.polar(torch.ones_like(phi), phi)
        K = self.prop.forward_field(p)
        amp = K.abs()
        u = K / (amp + 1e-8)
        s = ls_scale(roi_crop(amp, self.rois), roi_crop(t, self.rois)).clamp_min(1e-8)
        tgt_amp = t / s                                          # цель в масштабе поля
        k_gs = torch.where(self.mask, tgt_amp * u, K)
        k_res = torch.where(self.mask, (amp - tgt_amp) * u, torch.zeros_like(K))
        B = phi.shape[0]
        back = self.prop.backward_field(torch.cat([k_gs, k_res], 0)).sum(1)   # один вызов на оба поля
        h_gs, g = back[:B], back[B:]
        rms = h_gs.abs().pow(2).mean(dim=(-2, -1), keepdim=True).sqrt() + 1e-8
        rel = p.conj() * h_gs                                    # шаг GS относительно phi
        d_gs = _safe_angle(rel)
        rg = p.conj() * g / rms
        feat = torch.stack([torch.cos(d_gs), torch.sin(d_gs), (h_gs.abs() / rms).clamp(max=4.0),
                            (rg.real * 4).clamp(-30, 30), (rg.imag * 4).clamp(-30, 30),
                            torch.cos(phi), torch.sin(phi)], 1)       # (ограничение - защита от выбросов в fp16)
        return feat, d_gs

    def _block(self, i, phi, t):
        feat, d_gs = self._physics(phi, t)
        d = self.blocks[i](feat).float()[:, 0]
        return phi + self.alpha[i] * d_gs + torch.pi * torch.tanh(d)

    def forward_phases(self, tgt):
        t = tgt / (tgt.amax(dim=(-2, -1), keepdim=True) + 1e-6)
        phis = [self._init_phase(t)]
        for i in range(len(self.blocks)):
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                from torch.utils.checkpoint import checkpoint
                phi = checkpoint(self._block, i, phis[-1], t, use_reentrant=False)
            else:
                phi = self._block(i, phis[-1], t)
            phis.append(phi)
        return phis

    def forward_all(self, tgt):
        return [torch.polar(torch.ones_like(f), f) for f in self.forward_phases(tgt)]

    def forward(self, tgt):
        ps = self.forward_all(tgt)
        return ps[-1], ps[0]

    @torch.no_grad()
    def predict_phase(self, tgt):
        return self.forward_phases(tgt)[-1]

    def amplitudes(self, phasor):
        return self.prop.forward_field(phasor).abs()


def load_weights(model, state):
    """Загрузка весов; старые чекпойнты содержали константу prop.q - пропускаем её."""
    state = {k: v for k, v in state.items() if not k.endswith("prop.q")}
    model.load_state_dict(state)
    return model


def build_model(cfg):
    """Создаёт сеть по конфигу из чекпойнта (старые чекпойнты = v1)."""
    cfg = dict(cfg or {})
    arch = cfg.pop("arch", "v1")
    if arch == "v2":
        return UnrolledDOENet(**cfg)
    return DOENet(**cfg)

In [ ]:
%%writefile doe/data.py
"""Обучающие данные.

Готовые ДОЭ («правильные ответы») не нужны: сети достаточно целевых
изображений, а качество оценивается физической моделью. Поэтому датасет -
это просто много разных полутоновых картинок. Из «банка» фрагментов 64x64
на лету собираются четвёрки 62x62 и раскладываются по углам кадра 128x128
так же, как в проверочных изображениях.

Источники банка (--data):
  tinyimagenet  - 100k фото 64x64 с HuggingFace (нужен интернет, ~250 МБ) [по умолч.]
  cifar10       - 60k фото 32x32 через torchvision (увеличиваются до 64x64)
  folder:ПУТЬ   - любая папка с картинками (например, датасет Kaggle)
  synthetic     - процедурные картинки без скачивания (только для проверки кода)
  auto          - tinyimagenet -> cifar10 -> synthetic (первое, что сработает)
"""
import glob
import os

import numpy as np
import torch
from PIL import Image

from .physics import N, ROI, ROI_SIZE

BANK = 64


def _to_gray64(im):
    im = im.convert("L")
    if im.size != (BANK, BANK):
        w, h = im.size
        s = min(w, h)
        im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
        im = im.resize((BANK, BANK), Image.BICUBIC)
    return np.asarray(im, dtype=np.uint8)


def _bank_tinyimagenet(n_max):
    from datasets import load_dataset
    out = []
    for split in ("train", "valid"):
        ds = load_dataset("zh-plus/tiny-imagenet", split=split)
        for ex in ds:
            out.append(_to_gray64(ex["image"]))
            if len(out) >= n_max:
                return np.stack(out)
    return np.stack(out)


def _bank_cifar10(n_max, root):
    import torchvision
    out = []
    for train in (True, False):
        ds = torchvision.datasets.CIFAR10(root=root, train=train, download=True)
        for im, _ in ds:
            out.append(_to_gray64(im))
            if len(out) >= n_max:
                return np.stack(out)
    return np.stack(out)


def _bank_folder(path, n_max, crops_per_image=8, seed=0):
    rng = np.random.default_rng(seed)
    files = []
    for ext in ("png", "jpg", "jpeg", "bmp", "tif", "tiff", "webp"):
        files += glob.glob(os.path.join(path, "**", f"*.{ext}"), recursive=True)
        files += glob.glob(os.path.join(path, "**", f"*.{ext.upper()}"), recursive=True)
    files = sorted(set(files))
    if not files:
        raise FileNotFoundError(f"нет картинок в {path}")
    rng.shuffle(files)
    out = []
    for f in files:
        try:
            im = Image.open(f).convert("L")
        except Exception:
            continue
        w, h = im.size
        for _ in range(crops_per_image if min(w, h) > 2 * BANK else 1):
            s = int(rng.integers(BANK, max(BANK + 1, min(w, h) + 1)))
            x, y = int(rng.integers(0, w - s + 1)), int(rng.integers(0, h - s + 1))
            out.append(np.asarray(im.crop((x, y, x + s, y + s)).resize((BANK, BANK), Image.BICUBIC), np.uint8))
        if len(out) >= n_max:
            break
    return np.stack(out[:n_max])


def _bank_synthetic(n_max, seed=0):
    """Процедурные картинки: гладкий шум разных масштабов + фигуры + градиенты."""
    g = torch.Generator().manual_seed(seed)
    out = []
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, BANK), torch.linspace(-1, 1, BANK), indexing="ij")
    bs = 512
    for i in range(0, n_max, bs):
        b = min(bs, n_max - i)
        img = torch.zeros(b, BANK, BANK)
        for s in (4, 8, 16, 32):
            noise = torch.rand(b, 1, s, s, generator=g)
            img += torch.nn.functional.interpolate(noise, size=BANK, mode="bicubic", align_corners=False)[:, 0] * torch.rand(b, 1, 1, generator=g)
        for _ in range(4):
            cx, cy, r = (torch.rand(3, b, 1, 1, generator=g) * torch.tensor([2, 2, 0.6]).view(3, 1, 1, 1) - torch.tensor([1, 1, -0.1]).view(3, 1, 1, 1))
            circ = (((xx - cx) ** 2 + (yy - cy) ** 2) < r ** 2).float()
            img += circ * (torch.rand(b, 1, 1, generator=g) * 2 - 1)
        ang = torch.rand(b, 1, 1, generator=g) * 6.28
        img += (xx * torch.cos(ang) + yy * torch.sin(ang)) * torch.rand(b, 1, 1, generator=g)
        lo, hi = img.amin((1, 2), keepdim=True), img.amax((1, 2), keepdim=True)
        out.append(((img - lo) / (hi - lo + 1e-6) * 255).byte().numpy())
    return np.concatenate(out)


def bank_source(source="auto", n_max=60000, cache_dir="data"):
    """Какой источник реально использован для банка (после build_bank)."""
    tag = source.replace(":", "_").replace("/", "_").replace("\\", "_")
    path = os.path.join(cache_dir, f"bank_{tag}_{n_max}.npy.source.txt")
    try:
        with open(path, encoding="utf-8") as f:
            return f.read().strip()
    except OSError:
        return source


def build_bank(source="auto", n_max=60000, cache_dir="data", verbose=True):
    os.makedirs(cache_dir, exist_ok=True)
    tag = source.replace(":", "_").replace("/", "_").replace("\\", "_")
    cache = os.path.join(cache_dir, f"bank_{tag}_{n_max}.npy")
    if os.path.exists(cache):
        return np.load(cache)
    order = ["tinyimagenet", "cifar10", "synthetic"] if source == "auto" else [source]
    last_err = None
    for src in order:
        try:
            if verbose:
                print(f"[data] собираю банк из «{src}» ...", flush=True)
            if src == "tinyimagenet":
                bank = _bank_tinyimagenet(n_max)
            elif src == "cifar10":
                bank = _bank_cifar10(n_max, cache_dir)
            elif src.startswith("folder:"):
                bank = _bank_folder(src[len("folder:"):], n_max)
            elif src == "synthetic":
                bank = _bank_synthetic(n_max)
            else:
                raise ValueError(f"неизвестный источник {src}")
            np.save(cache, bank)
            with open(cache + ".source.txt", "w", encoding="utf-8") as f:
                f.write(src)                     # какой источник реально сработал (для DATASET.md)
            if verbose:
                print(f"[data] банк: {bank.shape}, сохранён в {cache}")
            return bank
        except Exception as e:      # пробуем следующий источник
            last_err = e
            print(f"[data] «{src}» не получилось: {e!r}")
    raise RuntimeError(f"не удалось собрать данные: {last_err!r}")


class QuadSampler:
    """Генератор батчей прямо на GPU: случайные 4 фрагмента -> аугментации ->
    раскладка по углам. Возвращает (B, 4, 128, 128) амплитуды в [0, 1]."""

    def __init__(self, bank_u8, device, val_frac=0.02, seed=0):
        bank = torch.as_tensor(bank_u8)
        n_val = max(64, int(len(bank) * val_frac))
        perm = torch.randperm(len(bank), generator=torch.Generator().manual_seed(seed))
        self.train = bank[perm[n_val:]].to(device)
        self.val = bank[perm[:n_val]].to(device)
        self.device = device
        self.ar = torch.arange(ROI_SIZE, device=device)

    def _patches(self, src, n, g):
        idx = torch.randint(0, len(src), (n,), device=self.device, generator=g)
        oy = torch.randint(0, BANK - ROI_SIZE + 1, (n,), device=self.device, generator=g)
        ox = torch.randint(0, BANK - ROI_SIZE + 1, (n,), device=self.device, generator=g)
        p = src[idx[:, None, None], oy[:, None, None] + self.ar[None, :, None],
                ox[:, None, None] + self.ar[None, None, :]].float() / 255.0
        r = torch.rand(n, 6, device=self.device, generator=g)
        p = torch.where(r[:, 0, None, None] < 0.5, p.flip(-1), p)
        p = torch.where(r[:, 1, None, None] < 0.5, p.flip(-2), p)
        p = torch.where(r[:, 2, None, None] < 0.5, p.transpose(-1, -2), p)
        gamma = torch.exp((r[:, 3] - 0.5) * 0.9)[:, None, None]            # 0.64..1.57
        p = p.clamp(0, 1) ** gamma
        lo = (r[:, 4] * 0.25)[:, None, None]                               # подъём «чёрного»
        hi = (0.6 + 0.4 * r[:, 5])[:, None, None]
        p = lo + (hi - lo) * p
        return p

    def sample(self, batch, split="train", g=None):
        src = self.train if split == "train" else self.val
        p = self._patches(src, batch * 4, g).view(batch, 4, ROI_SIZE, ROI_SIZE)
        out = torch.zeros(batch, 4, N, N, device=self.device)
        for k, (r, c) in enumerate(ROI):
            out[:, k, r, c] = p[:, k]
        return out

    def fixed_val(self, n=128, seed=123):
        g = torch.Generator(device=self.device).manual_seed(seed)
        return self.sample(n, "val", g)

In [ ]:
%%writefile doe/classic.py
"""Классические итеративные методы (для сравнения) и градиентная доводка фазы."""
import math
import time

import torch

from .metrics import full_loss, ls_scale
from .physics import ROI, Fresnel, roi_crop


def _roi_mask(n_planes, n, rois, device):
    m = torch.zeros(n_planes, n, n, dtype=torch.bool, device=device)
    for k, (r, c) in enumerate(rois):
        m[k, r, c] = True
    return m


@torch.no_grad()
def gerchberg_saxton(tgt, prop: Fresnel, iters=200, rois=ROI, seed=0, callback=None):
    """Многоплоскостной алгоритм Герчберга-Сэкстона (вариант «global GS»):
    в каждой плоскости внутри информативной области модуль заменяется на целевой
    (с подобранным масштабом), вне её поле не трогаем; затем поля всех плоскостей
    возвращаются в плоскость ДОЭ, усредняются, и от суммы остаётся только фаза.
    tgt: (P, n, n) амплитуды в [0, 1]. Возвращает phi (n, n)."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    P, n, _ = tgt.shape
    phi = (torch.rand(1, n, n, generator=g) * 2 - 1).to(tgt.device) * math.pi
    mask = _roi_mask(P, n, rois, tgt.device)
    t = tgt.unsqueeze(0)
    for it in range(iters):
        k = prop.forward_field(torch.polar(torch.ones_like(phi), phi))      # (1,P,n,n)
        amp = k.abs()
        s = ls_scale(roi_crop(amp, rois), roi_crop(t, rois))                 # (1,P,1,1)
        new_amp = torch.where(mask, t / s.clamp_min(1e-8), amp)
        k = torch.polar(new_amp, k.angle())
        h = prop.backward_field(k).sum(1)                                    # (1,n,n)
        phi = h.angle()
        if callback is not None:
            callback(it, phi[0])
    return phi[0]


def optimize_phase(tgt, prop: Fresnel, iters=300, lr=0.05, init=None, rois=ROI, seed=0,
                   callback=None, w_eff=0.0, cosine=False, decay_frac=0.3, warmup=None, w_frame=0.0):
    """Прямая оптимизация фазы ДОЭ градиентным спуском (Adam) по той же функции
    потерь, что и у нейросети. init=None - случайная фаза (классический метод),
    init=phi_nn - доводка ответа нейросети. tgt: (P, n, n).
    warmup: шагов плавного разгона. По умолчанию 10 при готовом старте (первый шаг
    Adam сдвигает каждый пиксел почти на полный шаг и портит хороший ответ сети),
    0 - при случайном старте (классический метод остаётся как был)."""
    P, n, _ = tgt.shape
    if warmup is None:
        warmup = 10 if init is not None else 0
    if init is None:
        g = torch.Generator(device="cpu").manual_seed(seed)
        init = (torch.rand(n, n, generator=g) * 2 - 1) * math.pi
    phi = init.detach().clone().to(tgt.device).unsqueeze(0).requires_grad_(True)
    opt = torch.optim.Adam([phi], lr=lr)
    # cosine=True: первые (1 - decay_frac) шагов - постоянный шаг, затем он плавно
    # уменьшается до 2%: быстро в начале и точная сходимость в конце
    def _lr(i):
        w = min(1.0, (i + 1) / warmup) if warmup > 0 else 1.0
        if not cosine:
            return w
        start = (1 - decay_frac) * iters
        if i < start:
            return w
        return w * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * (i - start) / max(1.0, iters - start))))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr) if (cosine or warmup > 0) else None
    t = roi_crop(tgt.unsqueeze(0), rois)
    for it in range(iters):
        opt.zero_grad(set_to_none=True)
        loss, _ = full_loss(prop.forward_phase(phi), t, rois, w_eff, w_frame)
        loss.backward()
        opt.step()
        if sched is not None:
            sched.step()
        if callback is not None:
            callback(it, phi.detach()[0])
    return phi.detach()[0]


def timed(fn, *a, **kw):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = fn(*a, **kw)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return out, time.perf_counter() - t0


def polish_quantized(phi, tgt, prop: Fresnel, rounds=600, rois=ROI, w_eff=0.0, verbose=False, w_frame=0.0):
    """Дискретная доводка уже 8-битного ДОЭ (то, что реально будет сдано в png).

    Квантование фазы на 256 уровней - заметный источник ошибки (ИСС 0.9999 -> 0.998).
    Алгоритм: фаза округляется до уровней png; затем по градиенту выбираются k
    пикселов, где сдвиг на ±1 уровень обещает наибольший выигрыш; шаг принимается,
    только если ошибка действительно уменьшилась (k растёт при успехе, уменьшается
    при неудаче). Перед этим подбирается общий сдвиг фазы (физически ничего не
    меняет, но меняет то, как пикселы округляются). Возвращает фазу ровно на уровнях png."""
    step = 2 * math.pi / 255
    t = roi_crop(tgt.unsqueeze(0), rois)

    def to_phi(u):
        return u.float() * step - math.pi

    def loss_of(u):
        with torch.no_grad():
            return full_loss(prop.forward_phase(to_phi(u)[None]), t, rois, w_eff, w_frame)[0].item()

    def levels(p):                       # как phase_to_uint8, но на torch; 255 == 0 (это одна фаза)
        w = torch.remainder(p + math.pi, 2 * math.pi)
        return torch.round(w / step).long() % 255

    phi = phi.detach().to(tgt.device)
    best_u, best_l = None, float("inf")
    for c in torch.linspace(0, step, 9)[:-1].tolist():
        u = levels(phi + c)
        lo = loss_of(u)
        if lo < best_l:
            best_u, best_l = u, lo
    u, cur, start = best_u, best_l, best_l
    k, fails = 256, 0
    for r in range(rounds):
        ph = to_phi(u)[None].requires_grad_(True)
        lo, _ = full_loss(prop.forward_phase(ph), t, rois, w_eff, w_frame)
        g, = torch.autograd.grad(lo, ph)
        g = g[0].ravel()
        idx = torch.topk(g.abs(), k).indices
        un = u.clone().ravel()
        un[idx] = (un[idx] - torch.sign(g[idx]).long()) % 255
        un = un.view_as(u)
        ln = loss_of(un)
        if ln < cur:
            u, cur, fails = un, ln, 0
            k = min(4096, int(k * 1.5) + 1)
        else:
            fails += 1
            k = max(1, k // 2)
            if k == 1 and fails > 12:
                break
    if verbose:
        print(f"[polish] ошибка после квантования {start:.6f} -> {cur:.6f} за {r + 1} раундов")
    return to_phi(u)

In [ ]:
%%writefile doe/evaluate.py
"""Ввод-вывод и итоговая оценка: ДОЭ.png -> 4 восстановленных png + метрики."""
import csv
import json
import os

import numpy as np
from PIL import Image

from .metrics import pcc_np, ssim_global_np, ssim_np, ssim_skimage
from .physics import (DISTANCES, ROI_INCLUSIVE, ROI_LITERAL, phase_to_uint8,
                      reconstruct_numpy, roi_crop_np, uint8_to_phase)


def load_targets(folder=None, files=None):
    """Читает 4 одноканальных png 128x128 (1.png..4.png). Возвращает uint8 (4,128,128).
    По заданию картинки 8-битные одноканальные; на всякий случай RGB/палитра
    переводятся в оттенки серого, а 16-битные - делением на 257 (65535 -> 255)."""
    if files is None:
        files = [os.path.join(folder, f"{i}.png") for i in range(1, 5)]
    imgs = []
    for f in files:
        im = Image.open(f)
        if im.mode != "L":
            print(f"[load] {os.path.basename(f)}: режим {im.mode}, ожидался L (8 бит, 1 канал) - конвертирую")
        if im.mode in ("I;16", "I;16B", "I;16L", "I"):
            a = np.asarray(im).astype(np.float64)
            a = np.clip(np.round(a / 257.0 if a.max() > 255 else a), 0, 255).astype(np.uint8)
        else:
            a = np.asarray(im.convert("L"))
        assert a.shape == (128, 128), f"{f}: ожидается 128x128, получено {a.shape}"
        imgs.append(a)
    return np.stack(imgs)


def save_gray(path, arr_u8):
    Image.fromarray(np.asarray(arr_u8, dtype=np.uint8), mode="L").save(path)


def amplitude_to_uint8(amp, target_u8, roi, mode="fit"):
    """Перевод модуля поля в 8-битное изображение.
    fit: яркость подбирается методом наименьших квадратов по информативной
         области (как подбор экспозиции камеры); КК от этого не зависит.
    max: максимум модуля в информативной области -> 255 (без знания цели).
    frame: максимум модуля во всём кадре 128x128 -> 255 (как mat2gray; без знания цели).
    best: как fit, но «экспозиция» уточняется перебором (±20%) так, чтобы сумма
          ИСС (оконного и глобального со слайда) 8-битной картинки была
          максимальной (КК от этого не меняется)."""
    r, c = roi
    a = amp[r, c].astype(np.float64)
    if mode in ("fit", "best"):
        t = target_u8[r, c].astype(np.float64)
        s = (a * t).sum() / ((a * a).sum() + 1e-12)
        if mode == "best":
            cands = s * np.linspace(0.8, 1.2, 41)
            scores = []
            for x in cands:
                q = np.clip(np.round(a * x), 0, 255)
                scores.append(ssim_np(q, t) + ssim_global_np(q, t))
            s = cands[int(np.argmax(scores))]
    elif mode == "frame":
        s = 255.0 / (amp.max() + 1e-12)
    else:
        s = 255.0 / (a.max() + 1e-12)
    return np.clip(np.round(amp * s), 0, 255).astype(np.uint8)


def evaluate_doe(doe_u8, targets_u8, out_dir=None, norm="fit", fftshift=True, verbose=True):
    """Главная функция проверки. Восстановление выполняется из 8-битного ДОЭ
    (т.е. ровно из того, что будет выведено на ПВМС), по полной формуле Френеля."""
    phi = uint8_to_phase(doe_u8)
    amps = reconstruct_numpy(phi, DISTANCES, fftshift=fftshift)
    recs = np.stack([amplitude_to_uint8(amps[k], targets_u8[k], ROI_INCLUSIVE[k], norm)
                     for k in range(4)])
    rows = []
    for k in range(4):
        r_, c_ = ROI_INCLUSIVE[k]
        eff = float((amps[k][r_, c_] ** 2).sum() / (amps[k] ** 2).sum())
        row = {"image": k + 1, "z_mm": round(DISTANCES[k] * 1e3, 1), "efficiency": eff}
        for tag, rois in (("", ROI_INCLUSIVE), ("_literal61", ROI_LITERAL)):
            a = roi_crop_np(recs, rois)[k]
            t = roi_crop_np(targets_u8, rois)[k]
            row["pcc" + tag] = pcc_np(a, t)
            row["ssim" + tag] = ssim_np(a, t)                     # оконный, как skimage
            row["ssim_global" + tag] = ssim_global_np(a, t)       # формула со слайда 3
            row["ssim_matlab" + tag] = ssim_np(a, t, pad="replicate")
        sk = ssim_skimage(roi_crop_np(recs)[k], roi_crop_np(targets_u8)[k])
        if sk:
            row.update(sk)
        # те же метрики при двух других способах перевести |K| в 8 бит (независимо от norm):
        r_, c_ = ROI_INCLUSIVE[k]
        for tag, md in (("_fit", "fit"), ("_frame", "frame")):
            img = amplitude_to_uint8(amps[k], targets_u8[k], ROI_INCLUSIVE[k], md)
            row["pcc" + tag] = pcc_np(img[r_, c_], targets_u8[k][r_, c_])
            row["ssim" + tag] = ssim_np(img[r_, c_], targets_u8[k][r_, c_])
        rows.append(row)
    summary = {"pcc_mean": float(np.mean([r["pcc"] for r in rows])),
               "ssim_mean": float(np.mean([r["ssim"] for r in rows])),
               "ssim_global_mean": float(np.mean([r["ssim_global"] for r in rows])),
               "ssim_matlab_mean": float(np.mean([r["ssim_matlab"] for r in rows])),
               "ssim_fit_mean": float(np.mean([r["ssim_fit"] for r in rows])),
               "ssim_frame_mean": float(np.mean([r["ssim_frame"] for r in rows]))}
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        save_gray(os.path.join(out_dir, "doe.png"), doe_u8)
        for k in range(4):
            save_gray(os.path.join(out_dir, f"rec_{k + 1}.png"), recs[k])
        with open(os.path.join(out_dir, "metrics.json"), "w", encoding="utf-8") as f:
            json.dump({"per_image": rows, "summary": summary, "norm": norm}, f, indent=2, ensure_ascii=False)
        with open(os.path.join(out_dir, "metrics.csv"), "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)
        save_overview(os.path.join(out_dir, "overview.png"), targets_u8, doe_u8, recs, rows)
    if verbose:
        print_table(rows, summary)
    return rows, summary, recs


def print_table(rows, summary):
    print(f"{'изобр.':>6} {'z, мм':>6} {'КК':>8} {'ИСС':>8} {'ИСС-глоб':>9} {'ИСС-MATLAB':>10} {'эфф.':>6}")
    for r in rows:
        print(f"{r['image']:>6} {r['z_mm']:>6} {r['pcc']:8.4f} {r['ssim']:8.4f} {r['ssim_global']:9.4f} "
              f"{r['ssim_matlab']:10.4f} {r['efficiency']:6.1%}")
    print(f"{'сред.':>6} {'':>6} {summary['pcc_mean']:8.4f} {summary['ssim_mean']:8.4f} "
          f"{summary['ssim_global_mean']:9.4f} {summary['ssim_matlab_mean']:10.4f}")
    print("ИСС - оконный (11x11, как skimage); ИСС-глоб - по формуле со слайда (одно окно на всю область); "
          "ИСС-MATLAB - оконный с краями как в MATLAB ssim()")


def save_overview(path, targets, doe, recs, rows):
    """Сводная картинка: верхний ряд - исходные, нижний - восстановленные, справа ДОЭ."""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    fig, ax = plt.subplots(2, 5, figsize=(15, 6.2))
    for k in range(4):
        ax[0, k].imshow(targets[k], cmap="gray", vmin=0, vmax=255)
        ax[0, k].set_title(f"исходное {k + 1}")
        ax[1, k].imshow(recs[k], cmap="gray", vmin=0, vmax=255)
        ax[1, k].set_title(f"z={rows[k]['z_mm']} мм  КК={rows[k]['pcc']:.3f}  ИСС={rows[k]['ssim']:.3f}", fontsize=9)
    ax[0, 4].imshow(doe, cmap="twilight", vmin=0, vmax=255)
    ax[0, 4].set_title("ДОЭ (фаза, 0..255)")
    ax[1, 4].axis("off")
    for a in ax.ravel():
        a.set_xticks([])
        a.set_yticks([])
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


def phase_to_doe_u8(phi):
    return phase_to_uint8(np.asarray(phi))

In [ ]:
%%writefile train.py
"""Обучение нейросети синтеза ДОЭ.

Основной режим (Kaggle/Colab, GPU; развёрнутая сеть v2, сам подстраивает число
итераций под бюджет времени и сохраняется по ходу):
    python train.py --data auto --time_budget_h 10.5 --amp --out runs/main
Быстрая проверка кода на слабом ПК (CPU, 1-3 минуты):
    python train.py --data synthetic --n_images 2000 --iters 30 --bs 2 --base 8 --stage_base 8 --stages 3 --val_every 15 --out runs/smoke
Старая сеть (v1):  --arch v1

Качество в процессе обучения контролируется только на отложенной части
обучающего набора (2% картинок, сеть их не видит); по ней же выбирается best.pt.
"""
import argparse
import copy
import csv
import json
import math
import os
import time

import torch

import torch.nn as nn

from doe.data import QuadSampler, build_bank
from doe.metrics import full_loss
from doe.model import build_model, load_weights
from doe.physics import ROI, roi_crop

T_START = time.time()


def get_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="auto", help="auto | tinyimagenet | cifar10 | folder:ПУТЬ | synthetic")
    p.add_argument("--n_images", type=int, default=110000, help="размер банка картинок")
    p.add_argument("--data_dir", default="data")
    p.add_argument("--out", default="runs/main")
    # архитектура
    p.add_argument("--arch", default="v2", choices=["v1", "v2"])
    p.add_argument("--base", type=int, default=32, help="ширина начальной U-Net A")
    p.add_argument("--base_ref", type=int, default=24, help="v1: ширина U-Net B")
    p.add_argument("--stages", type=int, default=12, help="v2: число развёрнутых блоков с физикой")
    p.add_argument("--stage_base", type=int, default=32, help="v2: ширина U-Net в каждом блоке")
    p.add_argument("--depth", type=int, default=3, help="v2: глубина U-Net блоков")
    p.add_argument("--no_grad_ckpt", action="store_true", help="v2: не экономить память (быстрее, но больше VRAM)")
    # обучение
    p.add_argument("--iters", type=int, default=200000, help="максимум итераций (обычно ограничивает бюджет времени)")
    p.add_argument("--time_budget_h", type=float, default=0.0,
                   help="сколько часов можно учиться; число итераций подбирается автоматически (0 - выкл.)")
    p.add_argument("--bs", type=int, default=16)
    p.add_argument("--lr", type=float, default=6e-4)
    p.add_argument("--warmup", type=int, default=1000)
    p.add_argument("--ema", type=float, default=0.999, help="скользящее среднее весов (0 - выкл.)")
    p.add_argument("--aux", type=float, default=0.3, help="вес потерь на промежуточных ответах")
    p.add_argument("--w_eff", type=float, default=0.0, help="штраф за свет вне информативных областей")
    p.add_argument("--val_every", type=int, default=500)
    p.add_argument("--amp", action="store_true", help="смешанная точность для свёрток (быстрее на GPU)")
    p.add_argument("--resume", default="auto",
                   help="продолжить прерванное обучение: путь к last.pt; 'auto' - взять OUT/last.pt, если есть; '' - нет")
    p.add_argument("--init", default="", help="тёплый старт: взять только веса из чекпойнта (новое расписание)")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--debug_nan_at", type=int, default=0, help=argparse.SUPPRESS)   # для проверки восстановления
    return p.parse_args()


class _Phases(nn.Module):
    """Обёртка: forward() возвращает список фаз всех блоков (для DataParallel)."""

    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, t):
        return self.m.forward_phases(t)


def model_cfg(a):
    if a.arch == "v1":
        return {"arch": "v1", "base": a.base, "base_ref": a.base_ref}
    return {"arch": "v2", "base": a.base, "stage_base": a.stage_base, "stages": a.stages, "depth": a.depth,
            "grad_ckpt": not a.no_grad_ckpt}


def supervised_loss(model, outs, tgt_roi, aux, w_eff):
    """Глубокий контроль: итоговый ответ с весом 1, промежуточные - с весом
    до aux (линейно растёт к концу). Это помогает каждому блоку учиться."""
    L = len(outs)
    loss, m_final = 0.0, None
    for i, p in enumerate(outs):
        w = 1.0 if i == L - 1 else aux * (i + 1) / (L - 1)
        if w == 0:
            continue
        l, m = full_loss(model.amplitudes(p), tgt_roi, ROI, w_eff)
        loss = loss + w * l
        if i == L - 1:
            m_final = m
    return loss, m_final


@torch.no_grad()
def validate(model, val, w_eff):
    was_training = model.training
    model.eval()
    tot = {"loss": 0.0, "pcc": 0.0, "ssim": 0.0, "pcc0": 0.0}
    nb = 0
    for i in range(0, len(val), 32):
        t = val[i:i + 32]
        outs = model.forward_all(t)
        l, m = full_loss(model.amplitudes(outs[-1]), roi_crop(t), ROI, w_eff)
        _, m0 = full_loss(model.amplitudes(outs[0]), roi_crop(t), ROI, 0.0)
        tot["loss"] += l.item()
        tot["pcc"] += m["pcc"]
        tot["ssim"] += m["ssim"]
        tot["pcc0"] += m0["pcc"]
        nb += 1
    model.train(was_training)
    return {k: v / nb for k, v in tot.items()}


def main():
    a = get_args()
    torch.manual_seed(a.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda":
        torch.backends.cudnn.benchmark = True
    print(f"[train] устройство: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""))
    os.makedirs(a.out, exist_ok=True)
    with open(os.path.join(a.out, "args.json"), "w") as f:
        json.dump(vars(a), f, indent=2)

    bank = build_bank(a.data, a.n_images, a.data_dir)
    sampler = QuadSampler(bank, device, seed=a.seed)
    val = sampler.fixed_val(128)

    cfg = model_cfg(a)
    model = build_model(cfg).to(device)
    print(f"[train] архитектура {a.arch}, параметров: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)
    plan = {"total": a.iters}       # общее число итераций; может уточниться по бюджету времени

    def lr_lambda(i):
        warm = min(1.0, (i + 1) / max(1, a.warmup))
        frac = min(i, plan["total"]) / max(1, plan["total"])
        return warm * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * frac)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    use_amp = a.amp and device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    ema = copy.deepcopy(model).eval() if a.ema > 0 else model
    if ema is not model:                     # (при --ema 0 веса самой модели не замораживаем)
        for p_ in ema.parameters():
            p_.requires_grad_(False)

    start, best = 0, float("inf")
    resume = os.path.join(a.out, "last.pt") if a.resume == "auto" else a.resume
    if resume and os.path.exists(resume):
        ck = torch.load(resume, map_location=device)
        load_weights(model, ck.get("model_raw", ck["model"]))
        if ema is not model:
            load_weights(ema, ck["model"])
        opt.load_state_dict(ck["opt"])
        start, best = ck["iter"], ck.get("best", best)
        plan["total"] = ck.get("total", plan["total"])
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda, last_epoch=start - 1)
        print(f"[train] продолжаю {resume} с итерации {start} из {plan['total']}")
    elif a.init:
        ck = torch.load(a.init, map_location=device)
        old = dict(ck.get("cfg", {}))
        old.setdefault("arch", "v1")
        same = all(old.get(k) == v for k, v in cfg.items() if k != "grad_ckpt")
        if not same:
            print(f"[train] ВНИМАНИЕ: {a.init} - другая архитектура ({old}), тёплый старт пропущен, учу с нуля")
        else:
            load_weights(model, ck["model"])
            if ema is not model:
                load_weights(ema, ck["model"])
            print(f"[train] тёплый старт с весов {a.init} (итерация {ck.get('iter', '?')})")

    log_path = os.path.join(a.out, "log.csv")
    new_log = not os.path.exists(log_path) or start == 0
    log_f = open(log_path, "w" if new_log else "a", newline="")
    log = csv.writer(log_f)
    if new_log:
        log.writerow(["iter", "train_loss", "train_pcc", "train_ssim", "val_loss", "val_pcc", "val_ssim",
                      "val_pcc_init", "lr", "sec"])

    def save(it, path, full=True):
        """full=True - всё для продолжения (last.pt); False - только веса для работы (best.pt)."""
        ck = {"model": ema.state_dict(), "iter": it, "best": best, "cfg": cfg}
        if full:
            ck.update({"model_raw": model.state_dict(), "opt": opt.state_dict(), "total": plan["total"]})
        torch.save(ck, path + ".tmp")
        os.replace(path + ".tmp", path)      # атомарно: файл не испортится при обрыве

    # Несколько GPU (на Kaggle «T4 x2»): батч делится между картами.
    runner = _Phases(model)
    n_gpu = torch.cuda.device_count() if device == "cuda" else 0
    if n_gpu > 1:
        try:                                 # пробная итерация: если что-то не так - одна карта
            dp = nn.DataParallel(runner)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                ph = dp(sampler.sample(a.bs))
            sum(f.float().mean() for f in ph).backward()
            opt.zero_grad(set_to_none=True)
            runner = dp
            print(f"[train] использую {n_gpu} GPU (DataParallel), по {a.bs // n_gpu} примеров на карту")
        except Exception as e:  # noqa: BLE001
            opt.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            print(f"[train] несколько GPU не сработали ({type(e).__name__}: {e}); учусь на одной")

    t_loop = time.time()
    run = {"loss": 0.0, "pcc": 0.0, "ssim": 0.0}
    nrun = 0
    it = start
    measure_from = None
    nan_streak, nan_total, recoveries = 0, 0, 0
    while it < plan["total"]:
        if a.time_budget_h > 0:
            if measure_from is None and it - start == 20:
                measure_from = (it, time.time())
            if measure_from is not None and it - measure_from[0] == 200:
                spi = (time.time() - measure_from[1]) / 200
                left = a.time_budget_h * 3600 - (time.time() - T_START) - 600     # 10 мин запас на финал
                new_total = min(a.iters, it + int(left / (spi * 1.08)))        # +8% на валидации
                plan["total"] = max(new_total, it + 1)
                print(f"[train] {spi:.3f} с/итер. -> всего {plan['total']} итераций "
                      f"(~{(plan['total'] - it) * spi * 1.08 / 3600:.1f} ч)", flush=True)
            if time.time() - T_START > a.time_budget_h * 3600 - 300:
                print("[train] бюджет времени исчерпан, завершаю")
                break
        tgt = sampler.sample(a.bs)
        tr = roi_crop(tgt)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            phis = runner(tgt)
        outs = [torch.polar(torch.ones_like(f), f.float()) for f in phis]
        loss, m = supervised_loss(model, outs, tr, a.aux, a.w_eff)
        it += 1
        if a.debug_nan_at and a.debug_nan_at <= it < a.debug_nan_at + 15 and recoveries == 0:
            loss = loss * float("nan")
        if not torch.isfinite(loss):
            # Обычно причина - переполнение половинной точности (fp16, --amp). Одиночный
            # пропуск безвреден. Если подряд много - веса больше не учатся: восстанавливаемся.
            nan_streak += 1
            nan_total += 1
            if nan_streak <= 3 or nan_streak % 50 == 0:
                print(f"[train] итерация {it}: loss не число, пропускаю шаг (подряд {nan_streak}, всего {nan_total})",
                      flush=True)
            opt.zero_grad(set_to_none=True)
            sched.step()
            if nan_streak >= 10:
                recoveries += 1
                bad = any(not torch.isfinite(q).all() for q in model.parameters())
                with torch.no_grad():          # сглаженная копия весов (EMA) не пострадала
                    for pm, pe in zip(model.parameters(), ema.parameters()):
                        pm.copy_(pe)
                opt.state.clear()              # сбросить накопленные моменты Adam
                what = "веса восстановлены из сглаженной копии (EMA)" + (" - они были испорчены" if bad else "")
                if use_amp:
                    use_amp = False
                    scaler = torch.amp.GradScaler("cuda", enabled=False)
                    what += "; смешанная точность выключена, дальше fp32 (медленнее, но без переполнений)"
                else:
                    for i_ in range(len(sched.base_lrs)):
                        sched.base_lrs[i_] *= 0.5
                    what += "; шаг обучения уменьшен вдвое"
                print(f"[train] {nan_streak} шагов подряд loss не число -> {what}", flush=True)
                if a.time_budget_h > 0:
                    measure_from = (it, time.time())   # скорость изменилась - пересчитать план
                nan_streak = 0
            continue
        nan_streak = 0
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        if ema is not model:
            d = min(a.ema, (1 + it) / (10 + it))
            with torch.no_grad():
                for pe, pm in zip(ema.parameters(), model.parameters()):
                    pe.mul_(d).add_(pm.detach(), alpha=1 - d)
                for be, bm in zip(ema.buffers(), model.buffers()):
                    be.copy_(bm)
        run["loss"] += loss.item()
        run["pcc"] += m["pcc"]
        run["ssim"] += m["ssim"]
        nrun += 1

        if it % a.val_every == 0 or it == plan["total"]:
            v = validate(ema, val, a.w_eff)
            tr_m = {k: x / max(1, nrun) for k, x in run.items()}
            el = time.time() - t_loop
            print(f"[{it:>6}/{plan['total']}] loss {tr_m['loss']:.4f} | val КК {v['pcc']:.4f} ИСС {v['ssim']:.4f} "
                  f"(старт КК {v['pcc0']:.3f}) | lr {sched.get_last_lr()[0]:.1e} | {el / 60:.1f} мин", flush=True)
            log.writerow([it, tr_m["loss"], tr_m["pcc"], tr_m["ssim"], v["loss"], v["pcc"], v["ssim"], v["pcc0"],
                          sched.get_last_lr()[0], round(el, 1)])
            log_f.flush()
            run = {k: 0.0 for k in run}
            nrun = 0
            if v["loss"] < best:       # выбор модели - по отложенной выборке
                best = v["loss"]
                save(it, os.path.join(a.out, "best.pt"), full=False)
            save(it, os.path.join(a.out, "last.pt"))
    save(it, os.path.join(a.out, "last.pt"))
    log_f.close()
    best_path = os.path.join(a.out, "best.pt")
    if not os.path.exists(best_path):
        save(it, best_path, full=False)
    ck = torch.load(best_path, map_location=device)
    load_weights(ema, ck["model"])
    v = validate(ema, val, a.w_eff)
    print(f"[train] лучшая модель (итерация {ck['iter']}) на отложенной выборке, без доводки: "
          f"КК {v['pcc']:.4f}, ИСС {v['ssim']:.4f}")
    print(f"[train] готово за {(time.time() - T_START) / 3600:.2f} ч. Чекпойнт: {best_path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile synthesize.py
"""Синтез ДОЭ для 4 изображений и формирование всех результатов для экспертов.

    python synthesize.py --images ПАПКА_С_1-4.png --method best --out results
Для нескольких комплектов сразу удобнее run.py.

Методы (--method):
  nn          - только нейросеть (один проход, миллисекунды)
  nn+refine   - нейросеть + градиентная доводка (--refine шагов)
  best        - максимум качества: доводка с затухающим шагом из ответа сети и из
                --restarts случайных стартов, затем дискретная доводка уже
                8-битного ДОЭ (учёт квантования фазы png); выбирается лучший
                по метрикам; яркость png подбирается под максимум ИСС  [по умолч.]
  adam        - классическая оптимизация фазы с нуля (--iters шагов), для сравнения
  gs          - многоплоскостной Герчберг-Сэкстон (--iters шагов), для сравнения
Результат в --out: doe.png, rec_1..4.png, metrics.json/csv, overview.png, timing.json.
Работает и на CPU.
"""
import argparse
import json
import os
import time

import numpy as np
import torch

from doe.classic import gerchberg_saxton, optimize_phase, polish_quantized, timed
from doe.evaluate import evaluate_doe, load_targets
from doe.model import build_model, load_weights
from doe.physics import Fresnel, phase_to_uint8


def load_model(path, device):
    ck = torch.load(path, map_location=device, weights_only=True)
    model = build_model(ck.get("cfg", {})).to(device).eval()
    load_weights(model, ck["model"])
    return model


def _score(phi, targets_u8, norm, robust=False):
    _, s, _ = evaluate_doe(phase_to_uint8(phi.cpu().numpy()), targets_u8, norm=norm, verbose=False)
    if robust:          # устойчивый режим: лучший тот, у кого выше худший из двух способов нормировки
        return min(s["ssim_fit_mean"], s["ssim_frame_mean"]) + s["pcc_mean"], s
    return s["pcc_mean"] + s["ssim_mean"], s


def synthesize(targets_u8, method="best", ckpt=None, refine=1000, iters=300, lr=0.05, device="cpu",
               w_eff=0.0, restarts=2, norm="best", model=None, verbose=True, w_frame=0.0):
    t = torch.tensor(targets_u8 / 255.0, dtype=torch.float32, device=device)
    prop = Fresnel().to(device)
    timing = {}
    if method in ("nn", "nn+refine", "best"):
        if model is None:
            model = load_model(ckpt, device)
            model.predict_phase(t[None])                  # прогрев
        phi, timing["nn_sec"] = timed(lambda: model.predict_phase(t[None])[0])
        if method == "nn+refine" and refine > 0:
            phi, timing["refine_sec"] = timed(optimize_phase, t, prop, iters=refine, lr=lr, init=phi, w_eff=w_eff,
                                              w_frame=w_frame)
        elif method == "best":
            t0 = time.perf_counter()
            starts = [("нейросеть", phi)] + [(f"случайный {i + 1}", None) for i in range(restarts)]
            best = None
            for j, (name, init) in enumerate(starts):
                cand = optimize_phase(t, prop, iters=refine, lr=lr, init=init, seed=100 + j,
                                      w_eff=w_eff, cosine=True, w_frame=w_frame)
                _, s0 = _score(cand, targets_u8, norm)
                cand = polish_quantized(cand, t, prop, w_eff=w_eff, w_frame=w_frame)
                sc, s = _score(cand, targets_u8, norm, robust=w_frame > 0)
                if verbose:
                    print(f"[best] старт «{name}»: КК {s['pcc_mean']:.5f}  ИСС {s['ssim_mean']:.5f}  "
                          f"(до учёта квантования ИСС {s0['ssim_mean']:.5f})")
                if best is None or sc > best[0]:
                    best = (sc, cand, name)
            phi = best[1]
            timing["refine_sec"] = time.perf_counter() - t0
            timing["chosen_start"] = best[2]
    elif method == "adam":
        phi, timing["adam_sec"] = timed(optimize_phase, t, prop, iters=iters, lr=lr, w_eff=w_eff)
    elif method == "gs":
        phi, timing["gs_sec"] = timed(gerchberg_saxton, t, prop, iters=iters)
    else:
        raise ValueError(method)
    timing["total_sec"] = sum(v for v in timing.values() if isinstance(v, float))
    return phi.cpu().numpy(), timing


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--images", default="examples/kit01_photos")
    p.add_argument("--files", nargs=4, default=None, help="4 файла вместо папки (по порядку 1..4)")
    p.add_argument("--ckpt", default="weights/doe_unrolled_v2.pt")
    p.add_argument("--method", default="best", choices=["nn", "nn+refine", "best", "adam", "gs"])
    p.add_argument("--refine", type=int, default=2000, help="шагов доводки (nn+refine, best)")
    p.add_argument("--restarts", type=int, default=0, help="best: сколько дополнительных случайных стартов")
    p.add_argument("--iters", type=int, default=300, help="шагов для adam / gs")
    p.add_argument("--lr", type=float, default=0.05)
    p.add_argument("--norm", default=None, choices=["fit", "max", "best", "frame"],
                   help="яркость восстановленных png: frame (максимум кадра -> 255), fit (МНК), best (максимум ИСС), max (максимум области). "
                        "По умолчанию frame")
    p.add_argument("--w_eff", type=float, default=0.0, help="штраф за свет вне областей при доводке")
    p.add_argument("--w_frame", type=float, default=0.5,
                   help="вес ошибки при нормировке по всему кадру (устойчивость к нормировке у жюри; run.py: 0.5)")
    p.add_argument("--out", default="results")
    p.add_argument("--cpu", action="store_true")
    a = p.parse_args()
    device = "cuda" if torch.cuda.is_available() and not a.cpu else "cpu"
    norm = a.norm or ("frame" if a.w_frame > 0 or a.method == "nn" else "best")

    targets = load_targets(a.images, a.files)
    phi, timing = synthesize(targets, a.method, a.ckpt, a.refine, a.iters, a.lr, device, a.w_eff,
                             a.restarts, norm, w_frame=a.w_frame)
    doe_u8 = phase_to_uint8(phi)
    print(f"[synth] метод: {a.method}, устройство: {device}, время: " +
          ", ".join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}" for k, v in timing.items()))
    evaluate_doe(doe_u8, targets, out_dir=a.out, norm=norm)
    np.save(os.path.join(a.out, "doe_phase_float.npy"), phi)
    with open(os.path.join(a.out, "timing.json"), "w", encoding="utf-8") as f:
        json.dump({"method": a.method, "device": device, "norm": norm, **timing}, f, indent=2, ensure_ascii=False)
    print(f"[synth] результаты сохранены в {os.path.abspath(a.out)}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile run.py
"""Главная точка входа: синтез фазового ДОЭ для каждого комплекта из 4 изображений.

    python run.py --input ПАПКА_С_КОМПЛЕКТАМИ --output ПАПКА_РЕЗУЛЬТАТОВ [--mode best]

Вход (--input), любой из вариантов:
  * папка с подпапками-комплектами:  input/<комплект>/{4 файла .png}
  * одна папка ровно с 4 файлами .png (один комплект)
Файлы комплекта сортируются по имени «по-человечески» (1.png, 2.png, ..., 10.png),
изображение k соответствует расстоянию k: 24,1 / 25,3 / 26,5 / 27,7 мм.
Изображения: 128x128, 8 бит, оттенки серого; информативная часть картинки k - в углу k.

Выход (--output), для каждого комплекта папка <комплект>/:
  doe.png                  фазовый ДОЭ: PNG 128x128, 8 бит, grayscale; 0 = -pi, 255 = +pi
  rec_1.png .. rec_4.png   восстановленные изображения (модуль поля, Френель с одним БПФ),
                           нормированы на максимум всего кадра -> 255 (стандартно, без знания цели)
  metrics.csv / .json      SSIM и Pearson по информативным областям
и общие файлы summary.csv (по комплектам + среднее) и timing.json.

Режимы (--mode):
  fast      только нейросеть (один проход)
  balanced  нейросеть + 800 шагов устойчивой доводки + дискретная доводка 8-битного ДОЭ
  best      нейросеть + 2000 шагов устойчивой доводки + дискретная доводка 8-битного ДОЭ  [по умолчанию]
  max       нейросеть + 4000 шагов устойчивой доводки + дискретная доводка (ещё точнее, вдвое дольше)
  best_roi  прежний вариант: 1000 шагов без устойчивости - максимум SSIM, если яркость
            восстановления подбирается по информативной области, но плохо при нормировке по кадру

«Устойчивая» доводка оптимизирует ДОЭ сразу под два способа перевести модуль поля в 8 бит:
с подбором яркости по информативной области и с нормировкой на максимум всего кадра
(как mat2gray). Тогда результат не зависит от того, как именно нормирует проверяющий.
"""
import argparse
import csv
import json
import os
import re
import sys
import time

import numpy as np
import torch

from doe.evaluate import evaluate_doe, load_targets
from doe.physics import ROI, phase_to_uint8
from synthesize import load_model, synthesize

W_FRAME = 0.5   # вес ошибки «нормировка по всему кадру» (подобран на dataset/samples, см. docs/tuning.md)
MODES = {  # режим -> (метод synthesize, шагов доводки, запасных стартов, w_frame, нормировка png)
    "fast": ("nn", 0, 0, 0.0, "frame"),
    "balanced": ("best", 800, 0, W_FRAME, "frame"),
    "best": ("best", 2000, 0, W_FRAME, "frame"),
    "max": ("best", 4000, 0, W_FRAME, "frame"),
    "best_roi": ("best", 1000, 0, 0.0, "best"),
}
HERE = os.path.dirname(os.path.abspath(__file__))
DEFAULT_WEIGHTS = os.path.join(HERE, "weights", "doe_unrolled_v2.pt")


def natural_key(s):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", s)]


def pngs(folder):
    return sorted([f for f in os.listdir(folder) if f.lower().endswith(".png")], key=natural_key)


def find_kits(root):
    """Список (имя комплекта, [4 пути к png])."""
    own = pngs(root)
    if len(own) >= 4:
        if len(own) > 4:
            print(f"[run] в {root} {len(own)} png - беру первые 4 по имени: {own[:4]}")
        return [(os.path.basename(os.path.normpath(root)) or "kit", [os.path.join(root, f) for f in own[:4]])]
    kits = []
    for d in sorted(os.listdir(root), key=natural_key):
        p = os.path.join(root, d)
        if os.path.isdir(p):
            fs = pngs(p)
            if len(fs) >= 4:
                if len(fs) > 4:
                    print(f"[run] {d}: {len(fs)} png - беру первые 4 по имени: {fs[:4]}")
                kits.append((d, [os.path.join(p, f) for f in fs[:4]]))
            elif fs:
                print(f"[run] {d}: только {len(fs)} png (нужно 4) - пропускаю")
    return kits


def check_layout(name, targets):
    """Предупреждение, если информативная часть картинки k не в своём углу."""
    for k in range(4):
        inside = targets[k][ROI[k]].astype(float).sum()
        total = targets[k].astype(float).sum() + 1e-9
        if inside / total < 0.9:
            print(f"[run] ВНИМАНИЕ {name}: у изображения {k + 1} только {inside / total:.0%} яркости в его области "
                  f"{[(s.start, s.stop) for s in ROI[k]]}; метод рассчитан на раскладку по углам из задания")


def main():
    p = argparse.ArgumentParser(description="Нейросетевой синтез фазового ДОЭ (4 изображения -> 1 ДОЭ)")
    p.add_argument("--input", default="/data/input", help="папка с комплектами (или с 4 png)")
    p.add_argument("--output", default="/data/output")
    p.add_argument("--mode", default="best", choices=list(MODES))
    p.add_argument("--weights", default=DEFAULT_WEIGHTS)
    p.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"])
    p.add_argument("--refine", type=int, default=None, help="переопределить число шагов доводки")
    p.add_argument("--threads", type=int, default=0, help="потоков CPU (0 - все)")
    p.add_argument("--no_extra", action="store_true", help="сохранять только doe.png (без восстановлений и метрик)")
    a = p.parse_args()

    if not os.path.isdir(a.input):
        demo = os.path.join(HERE, "examples")
        print(f"[run] папка {a.input} не найдена - запускаю на встроенных примерах {demo}")
        a.input = demo
    device = "cuda" if (a.device == "cuda" or (a.device == "auto" and torch.cuda.is_available())) else "cpu"
    if a.threads > 0:
        torch.set_num_threads(a.threads)
    method, refine, restarts, w_frame, norm = MODES[a.mode]
    if a.refine is not None:
        refine = a.refine
    kits = find_kits(a.input)
    if not kits:
        sys.exit(f"[run] в {a.input} не найдено ни одного комплекта из 4 png")
    os.makedirs(a.output, exist_ok=True)
    print(f"[run] комплектов: {len(kits)}, режим: {a.mode}, устройство: {device}"
          + (f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else f" ({torch.get_num_threads()} потоков CPU)"))

    t_load = time.perf_counter()
    model = load_model(a.weights, device)
    with torch.no_grad():                                     # прогрев (первый вызов всегда медленнее)
        model.predict_phase(torch.zeros(1, 4, 128, 128, device=device))
    t_load = time.perf_counter() - t_load

    rows, times = [], {}
    for name, files in kits:
        targets = load_targets(files=files)
        check_layout(name, targets)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        phi, _ = synthesize(targets, method, refine=refine, restarts=restarts, device=device, model=model,
                            verbose=False, w_frame=w_frame, norm=norm)
        doe_u8 = phase_to_uint8(phi)
        dt = time.perf_counter() - t0
        times[name] = dt
        out = os.path.join(a.output, name)
        os.makedirs(out, exist_ok=True)
        if a.no_extra:
            from doe.evaluate import save_gray
            save_gray(os.path.join(out, "doe.png"), doe_u8)
            print(f"[run] {name}: {dt:.2f} с")
            continue
        per, s, _ = evaluate_doe(doe_u8, targets, out_dir=out, norm=norm, verbose=False)
        row = {"kit": name, "ssim": s["ssim_mean"], "pearson": s["pcc_mean"],
               "ssim_frame_norm": s["ssim_frame_mean"], "ssim_roi_fit": s["ssim_fit_mean"],
               "ssim_global": s["ssim_global_mean"], "ssim_matlab": s["ssim_matlab_mean"]}
        if "ssim_skimage_default" in per[0]:
            row["ssim_skimage_default"] = float(np.mean([r["ssim_skimage_default"] for r in per]))
        row["time_sec"] = dt
        rows.append(row)
        print(f"[run] {name:24s} SSIM {row['ssim']:.4f}  Pearson {row['pearson']:.4f}  "
              f"(SSIM при нормировке по кадру {row['ssim_frame_norm']:.4f}, по области {row['ssim_roi_fit']:.4f})  "
              f"время {dt:.2f} с", flush=True)

    total = sum(times.values())
    if rows:
        keys = [k for k in rows[0] if k != "kit"]
        mean = {"kit": "СРЕДНЕЕ", **{k: float(np.mean([r[k] for r in rows])) for k in keys}}
        with open(os.path.join(a.output, "summary.csv"), "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["kit"] + keys)
            w.writeheader()
            w.writerows(rows + [mean])
        print(f"[run] СРЕДНЕЕ по {len(rows)} комплектам: SSIM {mean['ssim']:.4f}, Pearson {mean['pearson']:.4f}, "
              f"время {mean['time_sec']:.2f} с на комплект")
        print(f"[run]   SSIM при нормировке по кадру {mean['ssim_frame_norm']:.4f}, "
              f"при подборе яркости по области {mean['ssim_roi_fit']:.4f}")
        print("[run] SSIM - гауссово окно 11x11 (как skimage gaussian_weights=True); варианты по формуле со слайда, "
              "MATLAB и skimage по умолчанию - в summary.csv")
    with open(os.path.join(a.output, "timing.json"), "w", encoding="utf-8") as f:
        json.dump({"mode": a.mode, "device": device, "model_load_sec": t_load, "per_kit_sec": times,
                   "total_synthesis_sec": total, "mean_per_kit_sec": total / len(times)}, f, indent=2, ensure_ascii=False)
    print(f"[run] загрузка модели {t_load:.1f} с; синтез всех комплектов {total:.1f} с. Результаты: {os.path.abspath(a.output)}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile benchmark.py
"""Сравнение скорости и качества: нейросеть vs классические итеративные методы.

    python benchmark.py --images examples/kit01_photos --out bench

Строит кривые «качество от времени» и таблицу bench/table.md - это основной
аргумент на защите: во сколько раз нейросеть ускоряет синтез.
"""
import argparse
import json
import os
import time

import numpy as np
import torch

from doe.classic import gerchberg_saxton, optimize_phase
from doe.evaluate import evaluate_doe, load_targets
from doe.physics import Fresnel, phase_to_uint8
from synthesize import load_model


def score(phi, targets):
    _, s, _ = evaluate_doe(phase_to_uint8(phi.cpu().numpy()), targets, verbose=False)
    return s


def curve(run, targets, checkpoints):
    """Запускает итеративный метод, засекая время и качество на заданных итерациях."""
    pts = []
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    t_acc = [0.0]
    t_last = [time.perf_counter()]

    def cb(it, phi):
        sync()
        t_acc[0] += time.perf_counter() - t_last[0]
        if it + 1 in checkpoints:
            pts.append({"iter": it + 1, "sec": t_acc[0], **score(phi, targets)})
        t_last[0] = time.perf_counter()

    t_last[0] = time.perf_counter()
    run(cb)
    return pts


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--images", default="examples/kit01_photos")
    p.add_argument("--ckpt", default="weights/doe_unrolled_v2.pt")
    p.add_argument("--iters", type=int, default=500)
    p.add_argument("--out", default="bench")
    p.add_argument("--cpu", action="store_true")
    a = p.parse_args()
    device = "cuda" if torch.cuda.is_available() and not a.cpu else "cpu"
    os.makedirs(a.out, exist_ok=True)
    targets = load_targets(a.images)
    t = torch.tensor(targets / 255.0, dtype=torch.float32, device=device)
    prop = Fresnel().to(device)
    cps = sorted({c for c in (1, 2, 3, 5, 7, 10, 15, 20, 30, 40, 50, 75, 100, 150, 200, 300, 500, 1000, 2000)
                  if c <= a.iters} | {a.iters})
    res = {}
    gerchberg_saxton(t, prop, 3)          # прогрев (первый вызов медленнее)
    optimize_phase(t, prop, 3)

    print("[bench] Герчберг-Сэкстон ...")
    res["GS"] = curve(lambda cb: gerchberg_saxton(t, prop, a.iters, callback=cb), targets, cps)
    print("[bench] Adam со случайной фазы ...")
    res["Adam (случайный старт)"] = curve(lambda cb: optimize_phase(t, prop, a.iters, callback=cb), targets, cps)
    if os.path.exists(a.ckpt):
        model = load_model(a.ckpt, device)
        model.predict_phase(t[None])
        sync = torch.cuda.synchronize if device == "cuda" else (lambda: None)
        sync()
        t0 = time.perf_counter()
        phi_nn = model.predict_phase(t[None])[0]
        sync()
        t_nn = time.perf_counter() - t0
        res["НС (один проход)"] = [{"iter": 0, "sec": t_nn, **score(phi_nn, targets)}]
        print("[bench] НС + доводка ...")
        pts = curve(lambda cb: optimize_phase(t, prop, a.iters, init=phi_nn, callback=cb), targets, cps)
        for q in pts:
            q["sec"] += t_nn
        res["НС + доводка Adam"] = res["НС (один проход)"] + pts
    else:
        print(f"[bench] чекпойнт {a.ckpt} не найден - сравниваю только классические методы")

    with open(os.path.join(a.out, "bench.json"), "w", encoding="utf-8") as f:
        json.dump({"device": device, "results": res}, f, indent=2, ensure_ascii=False)

    # Таблица: сколько времени нужно каждому методу, чтобы достичь порога качества.
    lines = [f"Устройство: {device}\n", "| метод | итераций | время, с | КК (ср.) | ИСС (ср.) |", "|---|---|---|---|---|"]
    for name, pts in res.items():
        for q in pts:
            if q["iter"] in (0, 1, 5, 10, 20, 50, 100, 200, a.iters):
                lines.append(f"| {name} | {q['iter']} | {q['sec']:.3f} | {q['pcc_mean']:.4f} | {q['ssim_mean']:.4f} |")
    thrs = (0.80, 0.90, 0.95, 0.98)
    lines += ["", "Время до достижения порога ИСС (ср.):", "",
              "| метод | " + " | ".join(f"ИСС>={x:.2f}" for x in thrs) + " |", "|---" * (len(thrs) + 1) + "|"]
    for name, pts in res.items():
        cells = []
        for thr in thrs:
            hit = next((q for q in pts if q["ssim_mean"] >= thr), None)
            cells.append(f"{hit['sec']:.3f} с ({hit['iter']} ит.)" if hit else "не достигнут")
        lines.append(f"| {name} | " + " | ".join(cells) + " |")
    if "НС (один проход)" in res:
        nn_pt = res["НС (один проход)"][0]
        gs_eq = next((q for q in res["GS"] if q["ssim_mean"] >= nn_pt["ssim_mean"]), None)
        ad_eq = next((q for q in res["Adam (случайный старт)"] if q["ssim_mean"] >= nn_pt["ssim_mean"]), None)
        lines += ["", f"Один проход нейросети (ИСС {nn_pt['ssim_mean']:.4f}, {nn_pt['sec'] * 1e3:.1f} мс) по качеству равен: "
                  + (f"{gs_eq['iter']} итерациям GS ({gs_eq['sec'] * 1e3:.1f} мс)" if gs_eq else f"более чем {a.iters} итерациям GS")
                  + "; " + (f"{ad_eq['iter']} итерациям Adam ({ad_eq['sec'] * 1e3:.1f} мс)" if ad_eq
                            else f"более чем {a.iters} итерациям Adam") + "."]
    table = "\n".join(lines)
    print(table)
    with open(os.path.join(a.out, "table.md"), "w", encoding="utf-8") as f:
        f.write(table + "\n")

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
        for name, pts in res.items():
            x = [max(q["sec"], 1e-4) for q in pts]
            st = "o" if len(pts) == 1 else "-o"
            ax[0].plot(x, [q["pcc_mean"] for q in pts], st, label=name, ms=4)
            ax[1].plot(x, [q["ssim_mean"] for q in pts], st, label=name, ms=4)
        for a_, nm in zip(ax, ("КК Пирсона (ср.)", "ИСС / SSIM (ср.)")):
            a_.set_xscale("log")
            a_.set_xlabel(f"время синтеза, с ({device})")
            a_.set_ylabel(nm)
            a_.grid(alpha=0.3)
        ax[1].legend()
        fig.tight_layout()
        fig.savefig(os.path.join(a.out, "speed_vs_quality.png"), dpi=120)
    except ImportError:
        pass


if __name__ == "__main__":
    main()

In [ ]:
%%writefile export_dataset.py
"""Экспорт набора данных для экспертов («все реализованные материалы ... наборы данных»).

    python export_dataset.py --data auto --out dataset            # примеры + описание
    python export_dataset.py --data auto --out dataset --full     # + весь банк картинок (.npz)

Что получается в --out:
  DATASET.md            описание: источник, обработка, разбиение, как генерируются примеры
  samples/0001/1..4.png примеры входов сети ровно в формате проверки (128x128, 8 бит, 1 канал)
  samples_preview.png   мозаика из первых примеров
  bank_gray64.npz       (с --full) все исходные картинки 64x64 + индексы отложенной выборки
"""
import argparse
import os

import numpy as np
import torch

from doe.data import BANK, QuadSampler, bank_source, build_bank
from doe.evaluate import save_gray


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="auto")
    p.add_argument("--n_images", type=int, default=110000)
    p.add_argument("--data_dir", default="data")
    p.add_argument("--out", default="dataset")
    p.add_argument("--samples", type=int, default=50, help="сколько примеров-четвёрок сохранить как png")
    p.add_argument("--full", action="store_true", help="сохранить весь банк картинок (сотни МБ)")
    p.add_argument("--seed", type=int, default=0, help="тот же seed, что при обучении")
    a = p.parse_args()

    bank = build_bank(a.data, a.n_images, a.data_dir)
    sampler = QuadSampler(bank, "cpu", seed=a.seed)
    n_val = len(sampler.val)
    os.makedirs(os.path.join(a.out, "samples"), exist_ok=True)

    g = torch.Generator().manual_seed(2024)
    quads = (sampler.sample(a.samples, "train", g).numpy() * 255).round().astype(np.uint8)
    for i, q in enumerate(quads):
        d = os.path.join(a.out, "samples", f"{i + 1:04d}")
        os.makedirs(d, exist_ok=True)
        for k in range(4):
            save_gray(os.path.join(d, f"{k + 1}.png"), q[k])

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        rows = min(6, len(quads))
        fig, ax = plt.subplots(rows, 4, figsize=(8, 2 * rows))
        for i in range(rows):
            for k in range(4):
                ax[i, k].imshow(quads[i, k], cmap="gray", vmin=0, vmax=255)
                ax[i, k].set_xticks([])
                ax[i, k].set_yticks([])
                if i == 0:
                    ax[i, k].set_title(f"{k + 1}.png")
        fig.tight_layout()
        fig.savefig(os.path.join(a.out, "samples_preview.png"), dpi=90)
        plt.close(fig)
    except ImportError:
        pass

    if a.full:
        perm = torch.randperm(len(bank), generator=torch.Generator().manual_seed(a.seed)).numpy()
        np.savez_compressed(os.path.join(a.out, "bank_gray64.npz"), bank=bank,
                            val_indices=perm[:n_val], train_indices=perm[n_val:])

    real = bank_source(a.data, a.n_images, a.data_dir)
    descr = {
        "tinyimagenet": "Tiny-ImageNet (HuggingFace [`zh-plus/tiny-imagenet`](https://huggingface.co/datasets/zh-plus/tiny-imagenet), части `train` 100 000 + `valid` 10 000) - "
                        "подмножество ImageNet, 200 классов, цветные фото 64x64; используется для исследовательских "
                        "целей, как и сам ImageNet.",
        "cifar10": "CIFAR-10 (torchvision, 60 000 цветных фото 32x32, увеличены до 64x64 бикубически) - "
                   "запасной источник, если HuggingFace был недоступен.",
        "synthetic": "процедурные картинки (гладкий шум, круги, градиенты), без скачивания - "
                     "только для проверки кода, не для итогового обучения.",
    }.get(real, f"картинки из папки `{real[len('folder:'):]}`" if real.startswith("folder:") else real)
    text = f"""# Набор данных для обучения нейросети синтеза ДОЭ

## Что это
Сети не нужны готовые ДОЭ: она учится через физическую модель (дифракция Френеля),
поэтому набор данных - это только **изображения-цели**.

## Источник
* Фактический источник: **{real}** (параметр `--data {a.data}`); банк - **{len(bank)}** картинок.
* {descr}

## Обработка
1. Каждое фото переводится в оттенки серого (8 бит) и, если нужно, приводится к {BANK}x{BANK}.
2. Разбиение: 2% банка (**{n_val}** картинок) - отложенная выборка для выбора лучшей модели
   (сеть их не видит), остальное - обучение. Перемешивание с seed={a.seed}.
3. Обучающий пример (генерируется на лету, каждый раз новый):
   * 4 случайные картинки банка, случайный фрагмент 62x62;
   * случайные отражения / транспонирование, гамма 0.64-1.57, контраст и уровень чёрного;
   * картинка k кладётся в свой угол чёрного кадра 128x128 - ровно как в проверке:
     1: строки/столбцы [0:62, 0:62], 2: [0:62, 66:128], 3: [66:128, 0:62], 4: [66:128, 66:128].
   Число разных комбинаций практически бесконечно, поэтому переобучения нет.

## Файлы
* `samples/NNNN/1.png ... 4.png` - {a.samples} примеров обучающих входов в формате задания.
* `samples_preview.png` - мозаика из первых примеров.
* `bank_gray64.npz` - {"весь банк (`bank`: uint8 [N,64,64], `val_indices`, `train_indices`)" if a.full else "не сохранён (запустите с `--full`)"}.

## Как воспроизвести
```
python export_dataset.py --data {a.data} --n_images {a.n_images} --out {a.out} {"--full" if a.full else ""}
```
"""
    with open(os.path.join(a.out, "DATASET.md"), "w", encoding="utf-8") as f:
        f.write(text)
    print(f"[dataset] готово: {os.path.abspath(a.out)} (банк {bank.shape}, примеров {a.samples})")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile selfcheck.py
"""Самопроверка проекта (~1 мин на CPU):  python selfcheck.py

Проверяет физическую модель, кодирование фазы в png, метрики (против numpy и
scikit-image, если он установлен), модели и формат выходных файлов.
"""
import math
import os
import sys
import tempfile

import numpy as np
import torch
from PIL import Image

from doe.classic import gerchberg_saxton, optimize_phase, polish_quantized
from doe.evaluate import evaluate_doe, load_targets
from doe.metrics import pcc_np, ssim_global_np, ssim_np
from doe.model import DOENet, UnrolledDOENet, build_model, load_weights
from doe.physics import (DISTANCES, ROI, Fresnel, chirp_nyquist_ratio, output_pixel_pitch, phase_to_uint8,
                         reconstruct_numpy, uint8_to_phase)

HERE = os.path.dirname(os.path.abspath(__file__))
ok_all = True


def check(name, cond, info=""):
    global ok_all
    ok_all &= bool(cond)
    print(f"[{'OK' if cond else 'ОШИБКА'}] {name}" + (f"  ({info})" if info else ""))


torch.manual_seed(0)
print("Параметры задачи:")
for z in DISTANCES:
    print(f"  z = {z * 1e3:.1f} мм: шаг в плоскости изображения {output_pixel_pitch(z) * 1e6:.2f} мкм, "
          f"λz/(NΔx²) = {chirp_nyquist_ratio(z):.3f}")
check("квадратичная фаза дискретизована без наложения (λz/(NΔx²) ≥ 1)", all(chirp_nyquist_ratio(z) >= 1 for z in DISTANCES))

# --- физика --------------------------------------------------------------------
prop = Fresnel()
phi = (torch.rand(128, 128) * 2 - 1) * math.pi
a_t = prop.forward_phase(phi[None])[0].double().numpy()
a_n = reconstruct_numpy(phi.double().numpy())
c = (a_t * a_n).sum((1, 2)) / (a_n * a_n).sum((1, 2))
err = np.abs(a_n * c[:, None, None] - a_t).max() / a_t.max()
check("быстрая (torch) и полная (numpy, формула со слайда) модели совпадают", err < 1e-5, f"отн. ошибка {err:.1e}")
h = torch.polar(torch.ones_like(phi), phi)[None]
inv = (prop.backward_field(prop.forward_field(h)) - h[:, None]).abs().max().item()
check("обратное распространение - точная обратная операция", inv < 1e-5, f"{inv:.1e}")
check("центр кадра (нулевой порядок) в пикселе (64, 64), в зазоре между областями",
      np.unravel_index(np.argmax(np.abs(np.fft.fftshift(np.fft.fft2(np.ones((128, 128)))))), (128, 128)) == (64, 64))

# --- кодирование фазы ------------------------------------------------------------
u = phase_to_uint8(np.array([-math.pi, 0.0, math.pi - 1e-9]))
check("кодирование: -π -> 0, 0 -> 128, +π -> 255", [int(x) for x in u] == [0, 128, 255], str([int(x) for x in u]))
p = np.random.default_rng(0).uniform(-10, 10, (128, 128))
d = np.angle(np.exp(1j * (uint8_to_phase(phase_to_uint8(p)) - p)))
check("ошибка квантования фазы ≤ половины уровня (π/255)", np.abs(d).max() <= math.pi / 255 + 1e-9, f"{np.abs(d).max():.4f}")

# --- метрики -------------------------------------------------------------------
T = load_targets(os.path.join(HERE, "examples", "kit01_photos"))
A = T[3][ROI[3]].astype(float)
B = np.clip(A + np.random.default_rng(1).normal(0, 20, A.shape), 0, 255).round()
check("КК совпадает с numpy.corrcoef", abs(pcc_np(A, B) - np.corrcoef(A.ravel(), B.ravel())[0, 1]) < 1e-12)
A0 = A.copy(); pcc_np(A, B)
check("функция КК не портит входной массив", np.array_equal(A, A0))
check("ИСС(A, A) = 1 (оконный и глобальный)", abs(ssim_np(A, A) - 1) < 1e-12 and abs(ssim_global_np(A, A) - 1) < 1e-12)
try:
    from skimage.metrics import structural_similarity as ss
    ref = ss(A, B, data_range=255, gaussian_weights=True, sigma=1.5, use_sample_covariance=False)
    check("оконный ИСС совпадает с scikit-image", abs(ssim_np(A, B) - ref) < 1e-9, f"{ssim_np(A, B):.6f} vs {ref:.6f}")
except ImportError:
    print("[--] scikit-image не установлен - сверка ИСС пропущена")

# --- входные картинки -----------------------------------------------------------
check("пример комплекта: 4 шт., 128x128, uint8", T.shape == (4, 128, 128) and T.dtype == np.uint8)
outside = [T[k].copy() for k in range(4)]
for k in range(4):
    outside[k][ROI[k]] = 0
check("вне информативных областей картинки чёрные", all(o.max() == 0 for o in outside))

# --- модели --------------------------------------------------------------------
t = torch.tensor(T / 255.0, dtype=torch.float32)
m2 = UnrolledDOENet(base=8, stage_base=8, stages=2).eval()
with torch.no_grad():
    phis = m2.forward_phases(t[None])
    # необученный блок (поправка 0, alpha 1) должен быть ровно шагом GS
    K = prop.forward_field(torch.polar(torch.ones_like(phis[0]), phis[0]))
    from doe.metrics import ls_scale
    from doe.physics import roi_crop
    amp = K.abs()
    s = ls_scale(roi_crop(amp), roi_crop(t[None])).clamp_min(1e-8)
    kg = torch.where(m2.mask, (t[None] / s) * K / (amp + 1e-8), K)
    gs1 = prop.backward_field(kg).sum(1).angle()
dd = torch.angle(torch.exp(1j * (phis[1] - gs1))).abs().max().item()
check("необученный блок v2 = ровно одна итерация GS", dd < 1e-4, f"{dd:.1e}")
m2.train()
outs = m2.forward_all(t[None])
from doe.metrics import full_loss  # noqa: E402
lo, _ = full_loss(m2.amplitudes(outs[-1]), roi_crop(t[None]), ROI)
lo.backward()
check("градиенты v2 конечны", all(torch.isfinite(q.grad).all() for q in m2.parameters() if q.grad is not None))

m1 = DOENet(base=8, base_ref=8)
sd = m1.state_dict()
sd["prop.q"] = torch.zeros(4, 128, 128, dtype=torch.complex64)        # как в старых чекпойнтах
try:
    load_weights(build_model({"base": 8, "base_ref": 8}), sd)
    check("старые чекпойнты v1 (с prop.q) читаются", True)
except Exception as ex:  # noqa: BLE001
    check("старые чекпойнты v1 (с prop.q) читаются", False, str(ex))

# --- классика и итоговые файлы ---------------------------------------------------
phi_gs = gerchberg_saxton(t, prop, iters=30)
phi_ad = optimize_phase(t, prop, iters=60)
phi_po = polish_quantized(phi_ad, t, prop, rounds=40)
lv = (phi_po + math.pi) / (2 * math.pi / 255)
check("после дискретной доводки фаза ровно на уровнях png", (lv - lv.round()).abs().max().item() < 1e-3)
with tempfile.TemporaryDirectory() as td:
    rows, summ, _ = evaluate_doe(phase_to_uint8(phi_po.numpy()), T, out_dir=td, verbose=False)
    files = ["doe.png", "rec_1.png", "rec_2.png", "rec_3.png", "rec_4.png", "metrics.csv", "metrics.json"]
    check("все файлы результата созданы", all(os.path.exists(os.path.join(td, f)) for f in files))
    fmt = all(Image.open(os.path.join(td, f)).mode == "L" and Image.open(os.path.join(td, f)).size == (128, 128)
              for f in files[:5])
    check("png: 128x128, 8 бит, один канал (режим L)", fmt)
    back = np.asarray(Image.open(os.path.join(td, "doe.png")))
    check("doe.png читается обратно без потерь", np.array_equal(back, phase_to_uint8(phi_po.numpy())))
print(f"GS 30 итер.: ИСС {evaluate_doe(phase_to_uint8(phi_gs.numpy()), T, verbose=False)[1]['ssim_mean']:.3f}; "
      f"Adam 60 + доводка: ИСС {summ['ssim_mean']:.3f}")
print("\nИТОГ:", "всё в порядке" if ok_all else "ЕСТЬ ОШИБКИ")
sys.exit(0 if ok_all else 1)

In [ ]:
%%writefile scripts/make_example_kits.py
"""Собирает независимые тестовые комплекты examples/kitNN/1..4.png.

Комплекты НЕ используются при обучении сети (она училась на Tiny-ImageNet).
Источники: стандартные изображения из библиотеки scikit-image (`skimage.data`,
свободные изображения) и синтетические объекты, нарисованные этим скриптом.
Формат - ровно как в задании: 128x128, 8 бит, оттенки серого; картинка k лежит
в своём углу чёрного кадра, информативная область 62x62:
    1: [0:62, 0:62]   2: [0:62, 66:128]   3: [66:128, 0:62]   4: [66:128, 66:128]

    python scripts/make_example_kits.py            # -> examples/
"""
import os
import sys

import numpy as np
from PIL import Image, ImageDraw, ImageFont

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from doe.physics import ROI  # noqa: E402

OUT = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "examples")
S = 62


def gray(a):
    a = np.asarray(a)
    if a.dtype == bool:
        a = a.astype(np.uint8) * 255
    if a.dtype != np.uint8:
        a = (255 * (a - a.min()) / (a.max() - a.min() + 1e-12)).astype(np.uint8)
    im = Image.fromarray(a)
    return im.convert("L")


def square(im, box=None):
    """Квадратный фрагмент (центр или заданный box в долях) -> 62x62."""
    w, h = im.size
    if box is None:
        s = min(w, h)
        x, y = (w - s) // 2, (h - s) // 2
        im = im.crop((x, y, x + s, y + s))
    else:
        x0, y0, x1, y1 = box
        im = im.crop((int(x0 * w), int(y0 * h), int(x1 * w), int(y1 * h)))
    return np.asarray(im.resize((S, S), Image.LANCZOS))


def draw(fn):
    im = Image.new("L", (S * 4, S * 4), 0)
    fn(ImageDraw.Draw(im), S * 4)
    return np.asarray(im.resize((S, S), Image.LANCZOS))


def text_img(txt, size):
    im = Image.new("L", (S * 4, S * 4), 15)
    d = ImageDraw.Draw(im)
    font = ImageFont.load_default(size=size)
    bb = d.textbbox((0, 0), txt, font=font)
    d.text(((S * 4 - (bb[2] - bb[0])) / 2 - bb[0], (S * 4 - (bb[3] - bb[1])) / 2 - bb[1]), txt, fill=235, font=font)
    return np.asarray(im.resize((S, S), Image.LANCZOS))


def blobs(seed, thr=0.5):
    rng = np.random.default_rng(seed)
    a = rng.random((16, 16))
    im = Image.fromarray((a * 255).astype(np.uint8)).resize((S, S), Image.BICUBIC)
    a = np.asarray(im) / 255.0
    return ((a > thr) * 230 + 10).astype(np.uint8)


def kits():
    import skimage.data as d
    yy, xx = np.mgrid[0:S, 0:S] / (S - 1)
    r = np.hypot(xx - 0.5, yy - 0.5)
    return {
        "kit01_photos": [square(gray(d.astronaut()), (0.35, 0.05, 0.75, 0.45)), square(gray(d.camera())),
                         square(gray(d.coffee())), square(gray(d.chelsea()))],
        "kit02_textures": [square(gray(d.brick())), square(gray(d.grass())), square(gray(d.gravel())),
                           square(gray(d.coins()))],
        "kit03_text": [text_img("DOE", 110), square(gray(d.text()), (0.0, 0.0, 0.45, 1.0)),
                       square(gray(d.page()), (0.0, 0.0, 0.5, 1.0)), text_img("128", 110)],
        "kit04_shapes": [
            draw(lambda g, n: g.ellipse((n * .2, n * .2, n * .8, n * .8), outline=255, width=n // 14)),
            draw(lambda g, n: g.rectangle((n * .22, n * .22, n * .78, n * .78), fill=220)),
            draw(lambda g, n: g.polygon([(n * .5, n * .1), (n * .9, n * .85), (n * .1, n * .85)], fill=200)),
            draw(lambda g, n: [g.ellipse((n * c - n * .12, n * c2 - n * .12, n * c + n * .12, n * c2 + n * .12), fill=255)
                               for c, c2 in ((.3, .3), (.7, .3), (.3, .7), (.7, .7))])],
        "kit05_binary": [square(gray(d.horse()).point(lambda v: 255 - v)), square(gray(d.shepp_logan_phantom())),
                         square(gray(d.checkerboard())), blobs(7)],
        "kit06_science": [square(gray(d.moon())), square(gray(d.hubble_deep_field()), (0.3, 0.3, 0.6, 0.6)),
                          square(gray(d.immunohistochemistry())), square(gray(d.retina()), (0.25, 0.25, 0.75, 0.75))],
        "kit07_mixed": [square(gray(d.rocket())), square(gray(d.cell())), square(gray(d.microaneurysms())),
                        square(gray(d.clock()))],
        "kit08_smooth": [(xx * 230 + 10).astype(np.uint8), (np.clip(1 - r * 1.6, 0, 1) * 240).astype(np.uint8),
                         ((np.sin(xx * 12) * 0.5 + 0.5) * 200 + 20).astype(np.uint8),
                         (np.clip(square(gray(d.camera())) * 0.25 + 90, 0, 255)).astype(np.uint8)],
    }


def main():
    for name, imgs in kits().items():
        dd = os.path.join(OUT, name)
        os.makedirs(dd, exist_ok=True)
        for k, a in enumerate(imgs):
            frame = np.zeros((128, 128), np.uint8)
            frame[ROI[k]] = np.asarray(a, np.uint8)
            Image.fromarray(frame, mode="L").save(os.path.join(dd, f"{k + 1}.png"))
        print("ok", name)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/export_weights.py
"""Готовит файл весов для репозитория из чекпойнта обучения.

    python scripts/export_weights.py runs/main/best.pt weights/doe_unrolled_v2.pt

Берутся только сглаженные веса (EMA) и конфигурация сети; веса сохраняются в
половинной точности (float16): файл вдвое меньше (~65 МБ, проходит лимит GitHub
100 МБ), а при загрузке они переводятся обратно в float32. Разница в качестве
на уровне 1e-4 по SSIM (проверено на examples/).
"""
import os
import sys

import torch


def main(src, dst):
    ck = torch.load(src, map_location="cpu")
    sd = {k: (v.half() if v.is_floating_point() else v) for k, v in ck["model"].items()}
    out = {"model": sd, "cfg": ck["cfg"], "iter": ck.get("iter"), "val_loss": ck.get("best"), "stored_dtype": "float16"}
    os.makedirs(os.path.dirname(os.path.abspath(dst)), exist_ok=True)
    torch.save(out, dst)
    print(f"[weights] {src} -> {dst}: {os.path.getsize(dst) / 1e6:.1f} МБ, итерация {out['iter']}, "
          f"val loss {out['val_loss']}")


if __name__ == "__main__":
    if len(sys.argv) != 3:
        sys.exit(__doc__)
    main(sys.argv[1], sys.argv[2])

## 2. Зависимости

In [ ]:
!pip -q install datasets scikit-image

## 3. Примеры комплектов и быстрая проверка кода (~3 мин)
Примеры (`examples/`) — 8 независимых комплектов из стандартных изображений scikit-image и синтетических
фигур. В обучении они не участвуют; нужны только для самопроверки и демонстрации после обучения.
При ошибке здесь ноутбук остановится.

In [ ]:
run('python scripts/make_example_kits.py')
run('python selfcheck.py')
run(f"python train.py --data synthetic --n_images 2000 --iters 30 --bs 4 --base 8 --stage_base 8 --stages 3 "
    f"--val_every 15 --resume '' --data_dir {DATA_DIR} --out /tmp/smoke")

## 4. Основное обучение (~`HOURS` часов)
* Каждые 500 итераций печатаются метрики на отложенной части Tiny-ImageNet (2%, сеть их не видит);
  по ним же выбирается лучшая модель `best.pt`.
* `best.pt` / `last.pt` сохраняются по ходу. При сбоях «loss не число» обучение восстанавливается само.

In [ ]:
prev = sorted(glob.glob('/kaggle/input/**/runs/main/best.pt', recursive=True))
INIT = f"--init {prev[-1]}" if prev else ""
print('Тёплый старт:', prev[-1] if prev else 'нет (обучение с нуля)')
ok = run(f"python train.py --data auto --time_budget_h {HOURS} --amp --data_dir {DATA_DIR} --out runs/main {INIT}",
         strict=False)
if not os.path.exists('runs/main/best.pt'):
    raise RuntimeError('обучение не дало best.pt - смотрите лог выше')
if not ok:
    print('ВНИМАНИЕ: обучение завершилось с ошибкой, но best.pt есть - продолжаю с ним')
run('python scripts/export_weights.py runs/main/best.pt weights/doe_unrolled_v2.pt')

## 5. Синтез ДОЭ на примерах и сравнение с классическими методами

In [ ]:
run('python run.py --input examples --output results_examples --mode fast', strict=False)
run('python run.py --input examples --output results_examples_best --mode best', strict=False)
run('python benchmark.py --images examples/kit01_photos --iters 500 --out bench', strict=False)
from IPython.display import Image, display
for f in ('results_examples_best/kit01_photos/overview.png', 'bench/speed_vs_quality.png'):
    if os.path.exists(f):
        display(Image(f))

## 6. Набор данных для экспертов
Описание и примеры обучающих входов (`dataset/`) идут в архив; весь банк картинок
(`dataset/bank_gray64.npz`, ~300 МБ) остаётся во вкладке Output отдельно.

In [ ]:
run(f'python export_dataset.py --data auto --data_dir {DATA_DIR} --out dataset --samples 50 --full', strict=False)

## 7. Архив для скачивания (вкладка Output → doe_results.zip)

In [ ]:
import zipfile
OUT_ZIP = '/kaggle/working/doe_results.zip' if os.path.exists('/kaggle/working') else '/content/doe_results.zip'
include = ['weights/doe_unrolled_v2.pt', 'runs/main/log.csv', 'runs/main/args.json',
           'results_examples', 'results_examples_best', 'bench',
           'dataset/DATASET.md', 'dataset/samples', 'dataset/samples_preview.png']
with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for item in include:
        if os.path.isdir(item):
            for dp, _, fs in os.walk(item):
                for f in fs:
                    z.write(os.path.join(dp, f))
        elif os.path.exists(item):
            z.write(item)
        else:
            print('нет', item)
print(OUT_ZIP, round(os.path.getsize(OUT_ZIP) / 1e6, 1), 'МБ')
try:
    from google.colab import files
    files.download(OUT_ZIP)
except ImportError:
    print('Готово: doe_results.zip во вкладке Output')